In [ ]:
!pip install radgraph -q
!pip install "transformers==4.36.2" -q
import os, sys, json, re, time, random, shutil, argparse, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from typing import List, Dict, Tuple, Optional
from tqdm import tqdm

warnings.filterwarnings('ignore')


os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
import torch.nn as nn

SEED = 9223
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)






if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    if n_gpus >= 2:
        DEVICE   = torch.device('cuda:1')  
        DEVICE_1 = torch.device('cuda:0')  
        print(f"✅ DUAL GPU MODE")
        print(f"   GPU 0: {torch.cuda.get_device_name(0)} — {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB → R2Gen")
        print(f"   GPU 1: {torch.cuda.get_device_name(1)} — {torch.cuda.get_device_properties(1).total_memory/1e9:.1f} GB → VG-SCRRG + BERT + DenseNet + RadGraph")
    else:
        DEVICE   = torch.device('cuda:0')
        DEVICE_1 = torch.device('cuda:0')
        print(f"GPU: {torch.cuda.get_device_name(0)} — single GPU mode")
    
    
    torch.backends.cudnn.benchmark = True          
    torch.backends.cuda.matmul.allow_tf32 = True   
    torch.backends.cudnn.allow_tf32 = True         
    print(f"   cuDNN benchmark: ON | TF32: ON | FP16 autocast: ON")
else:
    DEVICE = torch.device('cpu')
    DEVICE_1 = DEVICE
    print("WARNING: No GPU — running on CPU")

print(f"\nDevice layout:")
print(f"   DEVICE   (cuda:1): VG-SCRRG + BERT + DenseNet + RadGraph")
print(f"   DEVICE_1 (cuda:0): R2Gen (hardcoded .cuda() constraint)")
print(f"   Seed: {SEED}")
print(f"\n✅ Environment ready — DUAL GPU + FP16")

In [ ]:
from pathlib import Path


CHUNK01_DIR  = Path('/kaggle/input/datasets/muhammadaneeq786/mimic-cxr-chunk-01-tar')
MANIFEST_DIR = Path('/kaggle/input/datasets/muhammadaneeq786/mimic-cxr-master-manifest')
ANNOTATION   = '/kaggle/input/datasets/muhammadaneeq786/annotation/annotation.json'


VG_SCRRG_PATH = Path('/kaggle/input/datasets/muhammadaneeq786/best-vg-scrrg-v410-mimic/best_vg_scrrg_v410_mimic (1).pt')
MODULE2_DIR   = Path('/kaggle/input/datasets/muhammadaneeq786/module2-outputs')
MODULE3_DIR   = Path('/kaggle/input/datasets/muhammadaneeq786/module3-outputs')

OUTPUT_DIR = Path('/kaggle/working/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


for _d, _n in [
    (CHUNK01_DIR,       'CHUNK01_DIR  (mimic-cxr-chunk-01-tar)'),
    (MANIFEST_DIR,      'MANIFEST_DIR (mimic-cxr-master-manifest)'),
    (VG_SCRRG_PATH,     'VG_SCRRG_PATH'),
    (MODULE2_DIR,       'MODULE2_DIR  (module2-outputs)'),
    (MODULE3_DIR,       'MODULE3_DIR  (module3-outputs)'),
    (Path(ANNOTATION),  'annotation.json'),
]:
    assert _d.exists(), f'\n❌ MISSING: {_n} not found at {_d}'
    print(f'✅ {_n}')


def _find_csv(filename, *search_dirs):
    for d in search_dirs:
        p = Path(d) / filename
        if p.exists():
            return p
        hits = list(Path(d).rglob(filename))
        if hits:
            return hits[0]
    return None

def _find_col(df, candidates):
    for c in candidates:
        if c in df.columns: return c
    low = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in low: return low[c.lower()]
    return df.columns[0]


metadata_csv  = _find_csv('mimic-cxr-2.0.0-metadata.csv', CHUNK01_DIR, MANIFEST_DIR)
split_csv     = _find_csv('mimic-cxr-2.0.0-split.csv',    CHUNK01_DIR, MANIFEST_DIR)
chexpert_csv  = _find_csv('mimic-cxr-2.0.0-chexpert.csv', CHUNK01_DIR, MANIFEST_DIR)
chunk01_man   = _find_csv('manifest_chunk_01.csv',         CHUNK01_DIR)
master_csv    = _find_csv('master_manifest.csv',           MANIFEST_DIR)

for name, path in [('metadata', metadata_csv), ('split', split_csv),
                   ('chexpert', chexpert_csv),  ('chunk01_manifest', chunk01_man),
                   ('master_manifest', master_csv)]:
    assert path is not None, f'Required CSV not found: {name}'
    print(f'  ✅ {name}: {path.name}')


print('\nLoading CSVs...')
metadata_df = pd.read_csv(metadata_csv)
split_df    = pd.read_csv(split_csv)
chexpert_df = pd.read_csv(chexpert_csv)
chunk01_df  = pd.read_csv(chunk01_man)
master_df   = pd.read_csv(master_csv)

DCOL_META  = _find_col(metadata_df, ['dicom_id', 'DicomId', 'DICOM_ID'])
DCOL_SPLIT = _find_col(split_df,    ['dicom_id', 'DicomId'])
DCOL_C01   = _find_col(chunk01_df,  ['dicom_id', 'DicomId', 'dicom'])
SCOL_MAST  = _find_col(master_df,   ['study_id', 'StudyId', 'study'])


chunk01_dicoms = set(chunk01_df[DCOL_C01].astype(str).str.strip())
metadata_df[DCOL_META] = metadata_df[DCOL_META].astype(str).str.strip()

frontal_df = metadata_df[
    metadata_df['ViewPosition'].isin(['PA', 'AP']) &
    metadata_df[DCOL_META].isin(chunk01_dicoms)
].copy().rename(columns={DCOL_META: 'dicom_id'})

frontal_df['image_filename'] = frontal_df['dicom_id'] + '.jpg'
IMAGE_DIR = CHUNK01_DIR  


_sample_n = min(200, len(frontal_df))
_n_found = sum((IMAGE_DIR / f).exists() for f in frontal_df['image_filename'].iloc[:_sample_n])
if _n_found == 0:
    _jpgs = list(CHUNK01_DIR.rglob('*.jpg'))
    if _jpgs:
        IMAGE_DIR = _jpgs[0].parent
        print(f'  Auto-detected IMAGE_DIR: {IMAGE_DIR}')
print(f'Image check ({_sample_n} sample): {_n_found}/{_sample_n} found ({_n_found/_sample_n*100:.0f}%)')


split_df[DCOL_SPLIT] = split_df[DCOL_SPLIT].astype(str).str.strip()
frontal_df = frontal_df.merge(
    split_df[[DCOL_SPLIT, 'split']].rename(columns={DCOL_SPLIT: 'dicom_id'}),
    on='dicom_id', how='inner'
)


frontal_df['_vp'] = frontal_df['ViewPosition'].map({'PA': 0, 'AP': 1}).fillna(2)
frontal_df = frontal_df.sort_values(['study_id', '_vp'])
study_df = frontal_df.drop_duplicates('study_id', keep='first').copy()
study_df.drop(columns=['_vp'], errors='ignore', inplace=True)


chexpert_df['study_id'] = (chexpert_df['study_id'].astype(str).str.strip()
                           .str.replace(r'^s', '', regex=True))
study_df['study_id'] = study_df['study_id'].astype(str).str.strip()
study_df = study_df.merge(
    chexpert_df.drop(columns=['subject_id'], errors='ignore'),
    on='study_id', how='left'
)


master_df[SCOL_MAST] = (master_df[SCOL_MAST].astype(str).str.strip()
                        .str.replace(r'^s', '', regex=True))
study_df = study_df[study_df['study_id'].isin(set(master_df[SCOL_MAST]))].copy()


train_df    = study_df[study_df['split'] == 'train'   ].reset_index(drop=True)
val_subset  = study_df[study_df['split'] == 'validate'].reset_index(drop=True)
test_subset = study_df[study_df['split'] == 'test'    ].reset_index(drop=True)

assert len(test_subset) > 0, (
    f'STOP: test_subset is empty.\n'
    f'  study_df: {len(study_df)}, check chunk01/metadata/split consistency.'
)


print('\nLoading annotation.json for ground-truth reports...')
with open(ANNOTATION) as f:
    _ann = json.load(f)

_gt_lookup = {}
for split_key in ('train', 'val', 'test', 'validate'):
    for entry in _ann.get(split_key, []):
        sid = entry.get('study_id') or entry.get('id', '')
        report = entry.get('report') or entry.get('text', '')
        if sid and report:
            _gt_lookup[str(sid).strip()] = report.strip()

print(f'  annotation.json: {len(_gt_lookup):,} reports loaded')
assert len(_gt_lookup) > 0, 'annotation.json loaded but contains no study_id/report entries.'


def get_best_image(row):
    
    fn = row.get('image_filename') if hasattr(row, 'get') else getattr(row, 'image_filename', None)
    if fn and str(fn) not in ('', 'nan', 'None'):
        return str(fn)
    did = row.get('dicom_id') if hasattr(row, 'get') else getattr(row, 'dicom_id', None)
    return f'{did}.jpg' if did and str(did) not in ('', 'nan', 'None') else None

def get_report_text(row):
    
    sid = str(row.get('study_id') if hasattr(row, 'get') else getattr(row, 'study_id', '')).strip()
    return _gt_lookup.get(sid, '')


predictions_df = None
for csv_name in ['predictions_baseline.csv', 'predictions_with_gt.csv',
                 'predictions.csv', 'official_250_predictions.csv']:
    csv_path = MODULE2_DIR / csv_name
    if csv_path.exists():
        predictions_df = pd.read_csv(csv_path)
        print(f'\n✅ Module 2 predictions: {len(predictions_df)} reports (from {csv_name})')
        break
if predictions_df is None:
    print(f'⚠️  No predictions CSV found in {MODULE2_DIR}')


with open(MODULE3_DIR / 'radgraph_metrics.json') as f:
    baseline_rg_metrics = json.load(f)
with open(MODULE3_DIR / 'module3_analysis.json') as f:
    m3_analysis = json.load(f)

print(f'\n📊 Module 3 Baseline:')
print(f'   Total entities : {m3_analysis["total_entities"]}')
print(f'   RG_ER          : {baseline_rg_metrics["RG_ER"]:.4f}')


test_samples = []
for idx, row in test_subset.iterrows():
    img = get_best_image(row)
    report = get_report_text(row)
    if img:
        test_samples.append({
            'id': f'test_{idx}',
            'study_id': str(row['study_id']),
            'image_path': img,
            'report': report,
        })

print(f'\n📊 Dataset: train={len(train_df):,}, val={len(val_subset):,}, test={len(test_subset):,}')
print(f'   Test samples with images : {len(test_samples):,}')
print(f'   Test samples with GT text: {sum(1 for s in test_samples if s["report"]):,}')
print(f'   IMAGE_DIR: {IMAGE_DIR}')
print(f'\n✅ MIMIC-CXR Official (Chunk 01) loaded — same split as Module 1 and Module 2')

In [ ]:
!pip install torchxrayvision
import torchvision.transforms as T
from PIL import Image
from transformers import AutoModel, AutoTokenizer






class GaussianFeatureNoise(nn.Module):
    
    def __init__(self, sigma=0.15):
        super().__init__()
        self.sigma = sigma

    def forward(self, x):
        if self.training and self.sigma > 0:
            return x + torch.randn_like(x) * self.sigma
        return x


class SpatialTokenDropout(nn.Module):
    
    def __init__(self, p=0.15):
        super().__init__()
        self.p = p

    def forward(self, x):
        if not self.training or self.p == 0:
            return x
        mask = torch.bernoulli(torch.full((x.size(0), x.size(1), 1), 1 - self.p, device=x.device))
        return x * mask / (1 - self.p)


class VisualEvidenceScorer(nn.Module):
    
    def __init__(self, visual_dim=1024, text_dim=768, hidden_dim=512,
                 num_heads=8, num_regions=8, dropout=0.30):
        super().__init__()

        
        self.feature_noise = GaussianFeatureNoise(sigma=0.15)
        self.token_dropout = SpatialTokenDropout(p=0.15)

        
        self.visual_proj = nn.Sequential(
            nn.Linear(visual_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )

        self.region_embed = nn.Embedding(num_regions, hidden_dim)

        
        self.cross_attn_1 = nn.MultiheadAttention(
            hidden_dim, num_heads, dropout=0.15, batch_first=True
        )
        self.norm_attn_1 = nn.LayerNorm(hidden_dim)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )
        self.norm_ffn = nn.LayerNorm(hidden_dim)

        
        self.cross_attn_2 = nn.MultiheadAttention(
            hidden_dim, num_heads, dropout=0.15, batch_first=True
        )
        self.norm_attn_2 = nn.LayerNorm(hidden_dim)

        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, visual_features, text_embeddings, region_ids):
        
        visual_features = self.feature_noise(visual_features)
        V = self.visual_proj(visual_features)  
        V = self.token_dropout(V)

        T = self.text_proj(text_embeddings)    
        R = self.region_embed(region_ids)      

        query = (T + R).unsqueeze(1)           

        
        attn_out_1, _ = self.cross_attn_1(query, V, V)
        query = self.norm_attn_1(query + attn_out_1)
        ffn_out = self.ffn(query)
        query = self.norm_ffn(query + ffn_out)

        
        attn_out_2, attn_weights = self.cross_attn_2(
            query, V, V, need_weights=True
        )
        attended = self.norm_attn_2(query + attn_out_2)
        attended = attended.squeeze(1)

        element_wise = attended * T
        combined = torch.cat([attended, T, element_wise], dim=-1)
        logits = self.classifier(combined).squeeze(-1)

        return logits, attn_weights.squeeze(1)


print("Loading VG-SCRRG verifier (Module 1)...")
vg_scrrg = VisualEvidenceScorer(dropout=0.30).to(DEVICE)

vg_weights_path = VG_SCRRG_PATH
if vg_weights_path.exists():
    state_dict = torch.load(vg_weights_path, map_location=DEVICE)
    vg_scrrg.load_state_dict(state_dict, strict=True)
    print(f"✅ Loaded VG-SCRRG V4.1 (MIMIC-CXR): {vg_weights_path} → {DEVICE}")
else:
    raise FileNotFoundError(f"VG-SCRRG weights not found: {vg_weights_path}")

vg_scrrg.eval()
for p in vg_scrrg.parameters():
    p.requires_grad = False

n_params = sum(p.numel() for p in vg_scrrg.parameters())
print(f"   Parameters: {n_params:,}")
print(f"   Architecture: [â; T; â⊙T] → MLP (matches Module 1)")


print("\nLoading TorchXRayVision DenseNet-121...")
import torchxrayvision as xrv
import torch.serialization
torch.serialization.add_safe_globals([xrv.models.DenseNet])

original_load = torch.load
def patched_load(f, *args, **kwargs):
    kwargs['weights_only'] = False
    return original_load(f, *args, **kwargs)
torch.load = patched_load
try:
    visual_encoder = xrv.models.DenseNet(weights="densenet121-res224-all")
finally:
    torch.load = original_load

visual_encoder = visual_encoder.to(DEVICE)
visual_encoder.eval()
for p in visual_encoder.parameters():
    p.requires_grad = False
print(f"✅ Visual encoder loaded (FROZEN) → {DEVICE}")


print("\nLoading Bio_ClinicalBERT...")
bert_tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
bert_model = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT").to(DEVICE)
bert_model.eval()
for p in bert_model.parameters():
    p.requires_grad = False
print(f"✅ Text encoder loaded (FROZEN) → {DEVICE}")


vg_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

@torch.no_grad()
def extract_visual_features(image_path: str) -> torch.Tensor:
    
    img = Image.open(image_path).convert('L')
    img_tensor = vg_transform(img).unsqueeze(0).to(DEVICE)
    img_tensor = img_tensor * 2048 - 1024  

    with torch.amp.autocast(device_type='cuda'):
        features = visual_encoder.features(img_tensor)  
    B, C, H, W = features.shape
    features = features.view(B, C, H * W).permute(0, 2, 1).float()  
    return features.squeeze(0)  

@torch.no_grad()
def encode_entity(entity: str) -> torch.Tensor:
    
    inputs = bert_tokenizer(entity, return_tensors='pt', padding=True,
                            truncation=True, max_length=32).to(DEVICE)
    with torch.amp.autocast(device_type='cuda'):
        outputs = bert_model(**inputs)
    return outputs.last_hidden_state[:, 0, :].float()  


REGION_NAMES = {
    0: 'Right Lung', 1: 'Left Lung', 2: 'Heart/Cardiac', 3: 'Mediastinum',
    4: 'Bone/Skeleton', 5: 'Pleural', 6: 'Diaphragm', 7: 'Global/Other'
}

print(f"\n✅ VG-SCRRG verification pipeline ready (FP16 autocast enabled)")

In [ ]:
import gdown

R2GEN_DIR = Path("/kaggle/working/R2Gen")


if not R2GEN_DIR.exists():
    !git clone https://github.com/cuhksz-nlp/R2Gen.git {R2GEN_DIR}
    print("✅ R2Gen cloned")
else:
    print("✅ R2Gen exists")

sys.path.insert(0, str(R2GEN_DIR))
from modules.tokenizers import Tokenizer
from models.r2gen import R2GenModel


R2GEN_CKPT = Path("/kaggle/input/datasets/muhammadaneeq786/model-mimic-cxr/model_mimic_cxr.pth")
if not R2GEN_CKPT.exists():
    print("Downloading R2Gen MIMIC-CXR checkpoint...")
    gdown.download(
        "https://drive.google.com/uc?id=1lNl8Ip2RMOEY1bP-KZzSdSBs0vhB3yb_",
        str(R2GEN_CKPT), quiet=False
    )
print(f"✅ R2Gen checkpoint: {R2GEN_CKPT}")


ann_for_tok = {"train": [], "val": [], "test": []}
for split_name, df in [("train", train_df), ("val", val_subset), ("test", test_subset)]:
    for idx, row in df.iterrows():
        img = get_best_image(row)
        report = get_report_text(row)
        if img and report:
            ann_for_tok[split_name].append({
                "id": f"{split_name}_{idx}",
                "report": report,
                "image_path": [img],
            })

ANN_PATH_M4 = "/kaggle/input/datasets/muhammadaneeq786/annotation/annotation.json"





K_CANDIDATES = 3

args = argparse.Namespace(
    image_dir=str(IMAGE_DIR),
    ann_path=ANN_PATH_M4,
    dataset_name='mimic_cxr',          
    max_seq_length=100,                 
    threshold=10,                       
    visual_extractor='resnet101',
    visual_extractor_pretrained=True,
    d_model=512, d_ff=512, d_vf=2048, num_heads=8, num_layers=3, dropout=0.1,
    logit_layers=1, use_bn=0, drop_prob_lm=0.5, bos_idx=0, eos_idx=0, pad_idx=0,
    rm_num_slots=3, rm_num_heads=8, rm_d_model=512,
    sample_method='beam_search',
    beam_size=K_CANDIDATES,
    temperature=1.0, sample_n=1, group_size=1, output_logsoftmax=1,
    decoding_constraint=0, block_trigrams=1,
    seed=SEED, n_gpu=1, num_workers=2, batch_size=1,
)


tokenizer = Tokenizer(args)
vocab_size = len(tokenizer.token2idx)


ckpt = torch.load(R2GEN_CKPT, map_location='cpu')
ckpt_state = ckpt.get('state_dict', ckpt)

ckpt_vocab = None
for key in sorted(ckpt_state.keys()):
    if 'tgt_embed' in key and 'lut' in key:
        ckpt_vocab = ckpt_state[key].shape[0]
        break


class TokenizerWrapper:
    def __init__(self, tok, target):
        self.tok = tok
        self.idx2token = tok.idx2token
        self.token2idx = tok.token2idx
        self._target = target
    def get_vocab_size(self):
        return self._target
    def __len__(self):
        return self._target
    def __call__(self, report):
        return self.tok(report)
    def decode(self, ids):
        return self.tok.decode(ids)
    def decode_batch(self, ids_batch):
        return self.tok.decode_batch(ids_batch)
    def clean_report(self, report):
        return self.tok.clean_report(report)

wrapped_tok = TokenizerWrapper(tokenizer, ckpt_vocab if ckpt_vocab else vocab_size + 1)



r2gen_model = R2GenModel(args, wrapped_tok)
try:
    r2gen_model.load_state_dict(ckpt_state, strict=True)
    print("✅ R2Gen MIMIC-CXR loaded (strict=True)")
except RuntimeError:
    r2gen_model.load_state_dict(ckpt_state, strict=False)
    print("✅ R2Gen MIMIC-CXR loaded (strict=False)")

r2gen_model = r2gen_model.to(DEVICE_1)  
r2gen_model.eval()
for p in r2gen_model.parameters():
    p.requires_grad = False

print(f"\n✅ R2Gen MIMIC-CXR on {DEVICE_1} (FP32, no autocast)")
print(f"   Vocab: {ckpt_vocab}")
print(f"   Beam size: {K_CANDIDATES}")


import torchvision.transforms as transforms
r2gen_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

def load_study_r2gen(sample) -> torch.Tensor:
    
    img_path = sample['image_path']
    if isinstance(img_path, list):
        img_path = img_path[0]
    full_path = IMAGE_DIR / img_path
    try:
        img = Image.open(full_path).convert('RGB')
        return r2gen_transform(img).unsqueeze(0)  
    except Exception:
        return torch.zeros(1, 3, 224, 224)


@torch.no_grad()
def generate_k_candidates(sample: Dict) -> List[Dict]:
    
    images = load_study_r2gen(sample).to(DEVICE_1) 

    candidates = []
    seen_reports = set()

    
    output = r2gen_model(images, mode='sample')
    all_beams = r2gen_model.encoder_decoder.done_beams[0]

    for rank, beam in enumerate(all_beams[:K_CANDIDATES]):
        tokens = beam['seq'].cpu().numpy()
        report = tokenizer.decode(tokens).strip()
        if report and report not in seen_reports:
            seen_reports.add(report)
            candidates.append({
                'report': report,
                'log_prob': float(beam['p']),
                'rank': len(candidates),
                'strategy': 'beam',
            })

    
    dropout_modules = []
    for m in r2gen_model.encoder_decoder.modules():
        if isinstance(m, nn.Dropout):
            dropout_modules.append(m)
            m.training = True
    try:
        output_mc = r2gen_model(images, mode='sample')
        mc_beams = r2gen_model.encoder_decoder.done_beams[0]
        for beam in mc_beams[:2]:
            tokens = beam['seq'].cpu().numpy()
            report = tokenizer.decode(tokens).strip()
            if report and report not in seen_reports:
                seen_reports.add(report)
                candidates.append({
                    'report': report,
                    'log_prob': float(beam['p']),
                    'rank': len(candidates),
                    'strategy': 'mc_dropout',
                })
    except Exception:
        pass
    finally:
        for m in dropout_modules:
            m.training = False

    
    noise = torch.randn_like(images) * 0.15
    images_aug = images + noise
    try:
        output_aug = r2gen_model(images_aug, mode='sample')
        aug_beams = r2gen_model.encoder_decoder.done_beams[0]
        for beam in aug_beams[:1]:
            tokens = beam['seq'].cpu().numpy()
            report = tokenizer.decode(tokens).strip()
            if report and report not in seen_reports:
                seen_reports.add(report)
                candidates.append({
                    'report': report,
                    'log_prob': float(beam['p']),
                    'rank': len(candidates),
                    'strategy': 'tta',
                })
    except Exception:
        pass

    if not candidates:
        top1 = tokenizer.decode_batch(output.cpu().numpy())[0].strip()
        candidates.append({'report': top1, 'log_prob': 0.0, 'rank': 0, 'strategy': 'fallback'})

    return candidates












@torch.no_grad()
def generate_k_candidates_batch(
    samples: List[Dict],
    batch_size: int = 8,
) -> Dict[int, List[Dict]]:
    
    all_cands: Dict[int, List[Dict]] = {}

    from tqdm.auto import tqdm as _tqdm
    for batch_start in _tqdm(range(0, len(samples), batch_size), desc='Phase 1 (beam)', leave=False):
        batch = samples[batch_start : batch_start + batch_size]

        
        imgs, valid_local = [], []
        for local_i, s in enumerate(batch):
            try:
                img = load_study_r2gen(s)          
                imgs.append(img)
                valid_local.append(local_i)
            except Exception:
                glob_i = batch_start + local_i
                all_cands[glob_i] = [{'report': '', 'log_prob': 0.0,
                                       'rank': 0, 'strategy': 'failed'}]

        if not imgs:
            continue

        
        stacked = torch.cat(imgs, dim=0).to(DEVICE_1)  
        B = stacked.size(0)

        try:
            
            r2gen_model(stacked, mode='sample')

            for bi in range(B):
                glob_i = batch_start + valid_local[bi]
                try:
                    beams = r2gen_model.encoder_decoder.done_beams[bi]
                    cands, seen = [], set()
                    for rank, beam in enumerate(beams[:K_CANDIDATES]):
                        report = tokenizer.decode(
                            beam['seq'].cpu().numpy()
                        ).strip()
                        if report and report not in seen:
                            seen.add(report)
                            cands.append({
                                'report':   report,
                                'log_prob': float(beam['p']),
                                'rank':     len(cands),
                                'strategy': 'beam',
                            })
                    all_cands[glob_i] = cands or [{
                        'report': '', 'log_prob': 0.0,
                        'rank': 0, 'strategy': 'fallback'
                    }]
                except Exception:
                    all_cands[batch_start + valid_local[bi]] = [{
                        'report': '', 'log_prob': 0.0,
                        'rank': 0, 'strategy': 'fallback'
                    }]

        except Exception as _batch_err:
            
            del stacked  
            torch.cuda.empty_cache()
            for bi in range(B):
                glob_i = batch_start + valid_local[bi]
                if glob_i in all_cands:
                    continue
                try:
                    single = imgs[bi].to(DEVICE_1)  
                    r2gen_model(single, mode='sample')
                    beams = r2gen_model.encoder_decoder.done_beams[0]
                    cands, seen = [], set()
                    for rank, beam in enumerate(beams[:K_CANDIDATES]):
                        report = tokenizer.decode(
                            beam['seq'].cpu().numpy()
                        ).strip()
                        if report and report not in seen:
                            seen.add(report)
                            cands.append({
                                'report':   report,
                                'log_prob': float(beam['p']),
                                'rank':     len(cands),
                                'strategy': 'beam',
                            })
                    all_cands[glob_i] = cands or [{
                        'report': '', 'log_prob': 0.0,
                        'rank': 0, 'strategy': 'fallback'
                    }]
                except Exception:
                    all_cands[glob_i] = [{
                        'report': '', 'log_prob': 0.0,
                        'rank': 0, 'strategy': 'failed'
                    }]

    return all_cands

print("✅ generate_k_candidates_batch ready (batch_size=8, beam-only, ~8x faster)")

test_sample = test_samples[0]
test_cands = generate_k_candidates(test_sample)
print(f"\n📊 Test: {len(test_cands)} candidates for {test_sample['id']}")
for c in test_cands:
    print(f"   [{c['strategy']}] rank={c['rank']} logP={c['log_prob']:.3f} | {c['report'][:80]}...")

print(f"\n✅ R2Gen MIMIC-CXR candidate generation ready")

In [ ]:
from radgraph import RadGraph



RADGRAPH_GPU = DEVICE.index if DEVICE.type == 'cuda' else 0

print(f"Loading RadGraph-XL on cuda:{RADGRAPH_GPU}...")
radgraph = RadGraph(model_type="radgraph-xl", cuda=RADGRAPH_GPU)
print(f"✅ RadGraph-XL loaded → cuda:{RADGRAPH_GPU}")


import sys
_SRC_DIR = '/kaggle/input/datasets/muhammadaneeq786/src-new2/src' if Path('/kaggle/input/datasets/muhammadaneeq786/src-new2/src').exists() else str(Path.cwd() / 'src')
if _SRC_DIR not in sys.path:
    sys.path.insert(0, _SRC_DIR)

try:
    from regions import get_region_ids as map_anatomy_to_region, NUM_REGIONS
    print(f"✅ Imported unified region mapping from src/regions.py ({NUM_REGIONS} regions)")
except ImportError:
    print("⚠️ Could not import src/regions.py — using inline fallback")
    NUM_REGIONS = 8
    
    ANATOMY_REGION_TABLE = {
        'right lung': 0, 'right upper lobe': 0, 'right middle lobe': 0, 'right lower lobe': 0,
        'right hilum': 0, 'right hemithorax': 0,
        'left lung': 1, 'left upper lobe': 1, 'left lower lobe': 1, 'left hilum': 1,
        'left hemithorax': 1, 'lingula': 1,
        'heart': 2, 'cardiac': 2, 'cardiac silhouette': 2, 'cardiomediastinal silhouette': 2,
        'pericardial': 2, 'pericardium': 2, 'cardiomegaly': 2, 'atrium': 2, 'ventricle': 2,
        'mediastinum': 3, 'mediastinal': 3, 'cardiomediastinal': 3, 'aorta': 3, 'aortic': 3,
        'hilar': 3, 'hila': 3, 'vascular': 3, 'vasculature': 3, 'pulmonary vasculature': 3,
        'vascular pedicle': 3, 'trachea': 3, 'carina': 3,
        'spine': 4, 'thoracic spine': 4, 'osseous': 4, 'bone': 4, 'bony': 4,
        'rib': 4, 'ribs': 4, 'skeletal': 4, 'vertebral': 4, 'vertebra': 4,
        'clavicle': 4, 'shoulder': 4, 'sternum': 4, 'scapula': 4, 'fracture': 4, 'scoliosis': 4,
        'pleural': 5, 'pleura': 5, 'pleural space': 5, 'costophrenic': 5,
        'costophrenic angle': 5, 'pneumothorax': 5, 'effusion': 5, 'pleural effusion': 5,
        'diaphragm': 6, 'hemidiaphragm': 6, 'diaphragmatic': 6,
        'right hemidiaphragm': 6, 'left hemidiaphragm': 6,
        'tube': 7, 'catheter': 7, 'pacemaker': 7, 'port': 7, 'line': 7, 'drain': 7,
        'airway': 7, 'bronchus': 7, 'bronchi': 7, 'bronchial': 7,
    }
    BILATERAL_LUNG_TERMS = {'lung', 'lungs', 'pulmonary', 'parenchyma', 'lobes', 'bases', 'apices'}
    
    OBSERVATION_REGION_TABLE = {
        'opacity': [0, 1], 'opacities': [0, 1], 'consolidation': [0, 1],
        'infiltrate': [0, 1], 'atelectasis': [0, 1], 'pneumonia': [0, 1],
        'edema': [0, 1], 'pulmonary edema': [0, 1], 'fibrosis': [0, 1],
        'emphysema': [0, 1], 'nodule': [0, 1], 'mass': [0, 1],
        'granuloma': [0, 1], 'calcification': [0, 1], 'thickening': [0, 1],
        'density': [0, 1], 'lesion': [0, 1], 'congestion': [0, 1],
        'cardiomegaly': [2], 'effusion': [5], 'pleural effusion': [5],
        'pneumothorax': [5], 'fracture': [4], 'scoliosis': [4],
    }
    def map_anatomy_to_region(anatomy_text: str) -> List[int]:
        
        text_lower = anatomy_text.lower().strip()
        if text_lower in ANATOMY_REGION_TABLE:
            return [ANATOMY_REGION_TABLE[text_lower]]
        if any(t in text_lower for t in BILATERAL_LUNG_TERMS):
            return [0, 1]
        if text_lower in OBSERVATION_REGION_TABLE:
            return list(OBSERVATION_REGION_TABLE[text_lower])
        for term, region_id in ANATOMY_REGION_TABLE.items():
            if term in text_lower:
                return [region_id]
        for term, region_ids in OBSERVATION_REGION_TABLE.items():
            if term in text_lower:
                return list(region_ids)
        if 'right' in text_lower:
            return [0]
        if 'left' in text_lower:
            return [1]
        return []

def extract_entities_radgraph(text: str) -> List[Dict]:
    
    if not text or len(text.strip()) < 5:
        return []

    try:
        result = radgraph([text])
        annotations = result.get("0", {}).get("entities", {})

        entities = []
        for eid, ent in annotations.items():
            label = ent.get("label", "")
            tokens = ent.get("tokens", "")
            relations = ent.get("relations", [])

            entities.append({
                "id": eid,
                "text": tokens,
                "label": label,
                "relations": relations,
            })

        return entities
    except Exception as e:
        return []

print(f"✅ RadGraph entity extraction ready (cuda:{RADGRAPH_GPU})")

In [ ]:
from collections import Counter


BEAM_ALPHA_EVIDENCE = 0.70   
BEAM_ALPHA_LOGPROB  = 0.30   


MIN_RERANK_GAIN = 0.08


SENT_FUSION_EV_THRESHOLD = 0.35   
SENT_FUSION_SIM_MIN      = 0.50   
SENT_FUSION_EV_GAIN_MIN  = 0.05   
MAX_FUSIONS_PER_REPORT   = 2      


HEDGE_THRESHOLD = 0.42   
HEDGE_PHRASES = {
    'default': 'may suggest',
    'opacity': 'possible',
    'effusion': 'cannot exclude',
    'consolidation': 'findings may represent',
    'pneumonia': 'concerning for',
    'atelectasis': 'suggestive of',
    'cardiomegaly': 'borderline',
    'edema': 'possible',
    'nodule': 'questionable',
    'mass': 'cannot exclude',
}


REMOVE_THRESHOLD = 0.30   
MIN_SENTENCES    = 3


MAX_REVERIFICATION = 2   


ANATOMY_KEYWORDS = {
    'lung', 'lungs', 'pulmonary', 'heart', 'cardiac', 'cardiomediastinal',
    'mediastinum', 'mediastinal', 'pleural', 'effusion', 'pneumothorax',
    'rib', 'ribs', 'bone', 'bony', 'osseous', 'spine', 'spinal',
    'diaphragm', 'hemidiaphragm', 'aorta', 'aortic', 'hilum', 'hilar',
    'hila', 'vasculature', 'vascular', 'trachea', 'silhouette',
    'costophrenic', 'parenchyma', 'airspace', 'opacity', 'consolidation',
    'atelectasis', 'infiltrate', 'nodule', 'mass', 'cardiomegaly',
    'edema', 'congestion', 'clavicle', 'shoulder', 'sternum',
    'right', 'left', 'bilateral', 'upper', 'lower', 'middle', 'lobe',
    'base', 'bases', 'apex', 'apices', 'lingula',
}

def get_anatomy_tokens(text: str) -> set:
    
    tokens = set(re.findall(r'\b\w+\b', text.lower()))
    return tokens & ANATOMY_KEYWORDS



def compute_self_bleu1(candidate: str, anchor: str) -> float:
    
    if not candidate or not anchor:
        return 0.0
    cand_tokens = candidate.lower().split()
    anchor_tokens = anchor.lower().split()
    if not cand_tokens or not anchor_tokens:
        return 0.0
    anchor_counts = Counter(anchor_tokens)
    matches = 0
    for token in cand_tokens:
        if anchor_counts.get(token, 0) > 0:
            matches += 1
            anchor_counts[token] -= 1
    return matches / len(cand_tokens)



@torch.no_grad()
def score_entity_evidence(
    visual_features: torch.Tensor,
    entity_text: str,
    region_ids: List[int]
) -> float:
    
    if not region_ids:
        region_ids = [7]  
    entity_emb = encode_entity(entity_text)
    vf = visual_features.unsqueeze(0).to(DEVICE)
    scores = []
    for rid in region_ids:
        rid_tensor = torch.tensor([rid], device=DEVICE)
        logits, _ = vg_scrrg(vf, entity_emb, rid_tensor)
        score = torch.sigmoid(logits).item()
        scores.append(score)
    return max(scores)



def split_into_sentences(text: str) -> List[str]:
    text = re.sub(r'\bdr\.', 'dr', text, flags=re.IGNORECASE)
    text = re.sub(r'\bno\.', 'no', text, flags=re.IGNORECASE)
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]


def map_entities_to_sentences(
    sentences: List[str], scored_entities: List[Dict]
) -> List[Dict]:
    
    sent_info = []
    for si, sent in enumerate(sentences):
        sent_tokens = set(re.findall(r'\b\w+\b', sent.lower()))
        obs_scores = []
        for ent in scored_entities:
            if not ent.get("is_observation") or ent.get("score") is None:
                continue
            ent_tokens = set(re.findall(r'\b\w+\b', ent.get("text", "").lower()))
            overlap = {t for t in (ent_tokens & sent_tokens) if len(t) > 2}
            if overlap:
                obs_scores.append(ent["score"])
        sent_info.append({
            'idx': si,
            'text': sent,
            'entity_scores': obs_scores,
            'avg_evidence': np.mean(obs_scores) if obs_scores else None,
            'n_entities': len(obs_scores),
        })
    return sent_info



def sentence_fusion(
    winner_report: str,
    winner_entities: List[Dict],
    all_scored_cands: List[Dict],
) -> Tuple[str, int]:
    
    winner_sents = split_into_sentences(winner_report)
    if len(winner_sents) <= 2:
        return winner_report, 0

    winner_sent_info = map_entities_to_sentences(winner_sents, winner_entities)

    
    alt_pool = []
    for cand in all_scored_cands:
        if cand['report'] == winner_report:
            continue
        cand_sents = split_into_sentences(cand['report'])
        cand_entities = cand.get('entities', [])
        cand_sent_info = map_entities_to_sentences(cand_sents, cand_entities)
        for cs in cand_sent_info:
            if cs['avg_evidence'] is not None:
                alt_pool.append(cs)

    if not alt_pool:
        return winner_report, 0

    repaired = list(winner_sents)
    n_fused = 0
    used_alt_texts = set()  

    
    low_ev_sents = [
        wsi for wsi in winner_sent_info
        if wsi['avg_evidence'] is not None and wsi['avg_evidence'] < SENT_FUSION_EV_THRESHOLD
    ]
    low_ev_sents.sort(key=lambda x: x['avg_evidence'])

    for wsi in low_ev_sents:
        
        if n_fused >= MAX_FUSIONS_PER_REPORT:
            break

        orig_anatomy = get_anatomy_tokens(wsi['text'])
        best_alt_text = None
        best_ev = wsi['avg_evidence']

        for alt in alt_pool:
            if alt['avg_evidence'] is None:
                continue
            
            if alt['avg_evidence'] < best_ev + SENT_FUSION_EV_GAIN_MIN:
                continue
            
            alt_norm = alt['text'].strip().lower()
            if alt_norm in used_alt_texts:
                continue
            
            sim = compute_self_bleu1(alt['text'], wsi['text'])
            if sim < SENT_FUSION_SIM_MIN:
                continue
            
            alt_anatomy = get_anatomy_tokens(alt['text'])
            if orig_anatomy and alt_anatomy and not (orig_anatomy & alt_anatomy):
                continue  

            best_ev = alt['avg_evidence']
            best_alt_text = alt['text']

        if best_alt_text is not None:
            repaired[wsi['idx']] = best_alt_text
            used_alt_texts.add(best_alt_text.strip().lower())
            n_fused += 1

    if n_fused == 0:
        return winner_report, 0

    return ' '.join(repaired), n_fused



def apply_hedging(report: str, scored_entities: List[Dict]) -> Tuple[str, int]:
    
    sentences = split_into_sentences(report)
    hedged = list(sentences)
    n_hedged = 0

    for ent in scored_entities:
        if not ent.get("is_observation"):
            continue
        score = ent.get("score", 1.0)
        if score is None:
            continue
        
        if REMOVE_THRESHOLD <= score < HEDGE_THRESHOLD:
            text = ent.get("text", "").lower().strip()
            hedge = HEDGE_PHRASES.get(text, HEDGE_PHRASES['default'])
            for i, sent in enumerate(hedged):
                if text in sent.lower() and hedge.lower() not in sent.lower():
                    
                    idx_in_sent = sent.lower().find(text)
                    if idx_in_sent >= 0:
                        hedged[i] = sent[:idx_in_sent] + hedge + " " + sent[idx_in_sent:]
                        n_hedged += 1
                        break
    return ' '.join(hedged), n_hedged



def apply_conservative_correction(
    report: str, scored_entities: List[Dict]
) -> Tuple[str, int]:
    
    sentences = split_into_sentences(report)
    if not sentences or len(sentences) <= MIN_SENTENCES:
        return report, 0

    def tokenize_simple(text):
        return set(re.findall(r'\b\w+\b', text.lower()))

    to_remove = set()
    for si, sent in enumerate(sentences):
        sent_tokens = tokenize_simple(sent)
        sent_obs_scores = []
        for ent in scored_entities:
            if not ent.get("is_observation") or ent.get("score") is None:
                continue
            ent_tokens = tokenize_simple(ent.get("text", ""))
            overlap = {t for t in (ent_tokens & sent_tokens) if len(t) > 2}
            if overlap:
                sent_obs_scores.append(ent["score"])
        if not sent_obs_scores:
            continue
        if all(s < REMOVE_THRESHOLD for s in sent_obs_scores):
            to_remove.add(si)

    remaining = len(sentences) - len(to_remove)

    if remaining < MIN_SENTENCES:
        
        removal_list = sorted(to_remove)
        max_removable = len(sentences) - MIN_SENTENCES
        to_remove = set(removal_list[:max(max_removable, 0)])

    kept = [s for i, s in enumerate(sentences) if i not in to_remove]
    corrected = " ".join(kept)

    if not corrected.strip():
        return report, 0

    return corrected, len(to_remove)



def compute_scas_f1(results_list):
    
    scas_f1_scores = []
    for r in results_list:
        n_obs_b = max(r.get('n_obs_baseline', 1), 1)
        n_obs_c = max(r.get('n_obs_corrected', 1), 1)
        n_sup_c = r.get('n_supported_corrected', 0)

        precision_s = n_sup_c / n_obs_c
        recall_s = n_sup_c / n_obs_b

        if precision_s + recall_s > 0:
            f1 = 2 * precision_s * recall_s / (precision_s + recall_s)
        else:
            f1 = 0.0
        scas_f1_scores.append(f1)

    return float(np.mean(scas_f1_scores)) if scas_f1_scores else 0.0


print("✅ V3.5 algorithm defined — Dual-Track + SAFE Sentence Fusion + Hedging")
print(f"   BEAM scoring: {BEAM_ALPHA_EVIDENCE}·evidence + {BEAM_ALPHA_LOGPROB}·logprob")
print(f"   NON-BEAM gate: MIN_RERANK_GAIN={MIN_RERANK_GAIN}")
print(f"   Sentence fusion: sim≥{SENT_FUSION_SIM_MIN}, ev_thresh={SENT_FUSION_EV_THRESHOLD}, max={MAX_FUSIONS_PER_REPORT}/report")
print(f"   Hedging: τ_hedge={HEDGE_THRESHOLD} (borderline entities get uncertainty markers)")
print(f"   Removal: τ_del={REMOVE_THRESHOLD}, min_sentences={MIN_SENTENCES}")
print(f"   Re-verification: max T={MAX_REVERIFICATION} iterations")
print(f"   Guards: deduplication + anatomy overlap + max cap")

In [ ]:
import os, warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"           
warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client")

from torch.utils.data import Dataset, DataLoader





class CXRImageDataset(Dataset):
    
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]
        try:
            img = Image.open(p).convert('L')
            t = self.transform(img)
            t = t * 2048 - 1024  
            return p, t
        except Exception:
            return p, torch.zeros(1, 224, 224)


@torch.no_grad()
def precompute_visual_features_fast(
    samples: List[Dict], batch_size: int = 32, num_workers: int = 4
) -> Dict[str, torch.Tensor]:
    
    
    unique_paths = []
    seen = set()
    for s in samples:
        img_p = s['image_path']
        if isinstance(img_p, list):
            img_p = img_p[0]
        p = str(IMAGE_DIR / img_p)
        if p not in seen:
            seen.add(p)
            unique_paths.append(p)

    dataset = CXRImageDataset(unique_paths, vg_transform)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=True,
        shuffle=False,
        persistent_workers=True if num_workers > 0 else False,
    )

    cache = {}
    for paths_batch, batch_tensor in loader:
        
        batch_tensor = batch_tensor.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type='cuda'):
            feats = visual_encoder.features(batch_tensor)  
        B, C, H, W = feats.shape
        feats = feats.view(B, C, H * W).permute(0, 2, 1).float()  
        for j, p in enumerate(paths_batch):
            cache[p] = feats[j]

    return cache





@torch.no_grad()
def encode_entities_batch(entities_texts: List[str], batch_size: int = 128) -> torch.Tensor:
    if not entities_texts:
        return torch.empty(0, 768, device=DEVICE)
    all_embeddings = []
    for i in range(0, len(entities_texts), batch_size):
        batch = entities_texts[i:i+batch_size]
        inputs = bert_tokenizer(batch, return_tensors='pt', padding=True,
                                truncation=True, max_length=32).to(DEVICE)
        with torch.amp.autocast(device_type='cuda'):
            outputs = bert_model(**inputs)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].float()
        all_embeddings.append(cls_embeddings)
    return torch.cat(all_embeddings, dim=0)








@torch.no_grad()
def score_entities_batch(
    visual_features: torch.Tensor,
    entity_texts: List[str],
    region_ids_list: List[List[int]],
    entity_score_cache: Dict
) -> List[float]:
    uncached_indices = []
    uncached_texts = []
    uncached_regions = []
    results_map = {}

    for i, (text, regions) in enumerate(zip(entity_texts, region_ids_list)):
        if not regions:
            regions = [7]  
        cache_key = (text.lower(), tuple(sorted(set(regions))))
        if cache_key in entity_score_cache:
            results_map[i] = entity_score_cache[cache_key]
        else:
            uncached_indices.append(i)
            uncached_texts.append(text)
            uncached_regions.append(regions)

    if uncached_texts:
        
        all_embeddings = encode_entities_batch(uncached_texts)

        
        batched_vf = []
        batched_emb = []
        batched_rids = []
        pair_to_entity = []  

        for batch_idx, (orig_idx, text, regions) in enumerate(
            zip(uncached_indices, uncached_texts, uncached_regions)
        ):
            entity_emb = all_embeddings[batch_idx]
            for rid in set(regions):
                batched_vf.append(visual_features)       
                batched_emb.append(entity_emb)
                batched_rids.append(rid)
                pair_to_entity.append(batch_idx)

        
        if batched_vf:
            b_vf = torch.stack(batched_vf)                     
            b_emb = torch.stack(batched_emb)                
            b_rids = torch.tensor(batched_rids, device=DEVICE) 

            
            VG_BATCH_SIZE = 512
            all_scores = []
            for start in range(0, len(b_rids), VG_BATCH_SIZE):
                end = min(start + VG_BATCH_SIZE, len(b_rids))
                with torch.amp.autocast(device_type='cuda'):
                    logits, _ = vg_scrrg(
                        b_vf[start:end],
                        b_emb[start:end],
                        b_rids[start:end]
                    )
                    sub_scores = torch.sigmoid(logits).reshape(-1)
                    all_scores.append(sub_scores)

            all_scores = torch.cat(all_scores).cpu().tolist()
            if isinstance(all_scores, float):
                all_scores = [all_scores]

            
            entity_max_scores = {}
            for pair_idx, score in enumerate(all_scores):
                ent_idx = pair_to_entity[pair_idx]
                if ent_idx not in entity_max_scores or score > entity_max_scores[ent_idx]:
                    entity_max_scores[ent_idx] = score

            
            for batch_idx, (orig_idx, text, regions) in enumerate(
                zip(uncached_indices, uncached_texts, uncached_regions)
            ):
                score = entity_max_scores.get(batch_idx, 0.0)
                cache_key = (text.lower(), tuple(sorted(set(regions))))
                entity_score_cache[cache_key] = score
                results_map[orig_idx] = score

    return [results_map[i] for i in range(len(entity_texts))]








def extract_entities_batch_radgraph(texts: List[str]) -> List[List[Dict]]:
    
    if not texts:
        return [[] for _ in texts]
    text_dict = {str(i): t for i, t in enumerate(texts) if t and len(t.strip()) >= 5}
    if not text_dict:
        return [[] for _ in texts]
    try:
        raw_results = radgraph(list(text_dict.values()))
    except Exception:
        return [[] for _ in texts]
    all_entities = []
    text_idx = 0
    for i in range(len(texts)):
        if str(i) in text_dict:
            key = str(text_idx)
            annotations = raw_results.get(key, {}).get("entities", {})
            entities = []
            for eid, ent in annotations.items():
                entities.append({
                    "id": eid,
                    "text": ent.get("tokens", ""),
                    "label": ent.get("label", ""),
                    "relations": ent.get("relations", []),
                })
            all_entities.append(entities)
            text_idx += 1
        else:
            all_entities.append([])
    return all_entities


def preextract_all_radgraph(all_candidates, batch_size=64):
    
    
    all_texts = []
    seen = set()
    for idx, cands in all_candidates.items():
        for c in cands:
            t = c['report']
            if t and t not in seen:
                seen.add(t)
                all_texts.append(t)

    print(f"   RadGraph pre-extraction: {len(all_texts)} unique texts")

    
    text_to_entities = {}
    for i in range(0, len(all_texts), batch_size):
        batch = all_texts[i:i+batch_size]
        batch_entities = extract_entities_batch_radgraph(batch)
        for t, ents in zip(batch, batch_entities):
            text_to_entities[t] = ents

    print(f"   ✅ RadGraph pre-extraction done: {len(text_to_entities)} texts cached")
    return text_to_entities





print("RUNNING V3.5 — DUAL-TRACK + SAFE FUSION + HEDGING + RE-VERIFY")
print("⚡ GPU-OPTIMIZED: DataLoader, Vectorized VG-SCRRG, Global RadGraph")
print("=" * 70)
n_test = len(test_samples)
print(f"Test samples: {n_test}")
if torch.cuda.device_count() >= 2:
    print(f"GPU 0: R2Gen FP32 — {torch.cuda.get_device_name(0)}")
    print(f"GPU 1: VG-SCRRG+BERT+DenseNet+RadGraph FP16 — {torch.cuda.get_device_name(1)}")
else:
    print(f"Single GPU mode")
print(f"V3.5 Scoring: {BEAM_ALPHA_EVIDENCE}·ev + {BEAM_ALPHA_LOGPROB}·lp + hedge(τ={HEDGE_THRESHOLD}) + reverify(T={MAX_REVERIFICATION})")
print(f"Fusion: sim≥{SENT_FUSION_SIM_MIN}, ev_thresh={SENT_FUSION_EV_THRESHOLD}, max={MAX_FUSIONS_PER_REPORT}/report, anatomy guard")
print(f"Removal: thresh={REMOVE_THRESHOLD}, min_sent={MIN_SENTENCES}")
print()

total_start = time.time()







print("▶ PHASE 1: R2Gen candidates (cuda:0, FP32, BATCHED ⚡)...")
PHASE1_BATCH_SIZE = 1   
phase1_start = time.time()

all_candidates = generate_k_candidates_batch(
    test_samples, batch_size=PHASE1_BATCH_SIZE
)

failed_gen = sum(1 for c in all_candidates.values() if c[0]['strategy'] == 'failed')
phase1_time = time.time() - phase1_start
n_unique_avg = np.mean([len(c) for c in all_candidates.values()])
print(f"  ✅ Phase 1: {len(all_candidates)} studies, {n_unique_avg:.1f} avg candidates, {phase1_time:.1f}s")
if failed_gen > 0:
    print(f"     Failed: {failed_gen}")
torch.cuda.empty_cache()




print("\n▶ PHASE 2: Visual features (DenseNet, cuda:1, FP16, DataLoader)...")
phase2_start = time.time()
visual_cache = precompute_visual_features_fast(test_samples, batch_size=32, num_workers=4)
phase2_time = time.time() - phase2_start
print(f"  ✅ Phase 2: {len(visual_cache)} images cached, {phase2_time:.1f}s")




print("\n▶ PHASE 2.5: Global RadGraph pre-extraction...")
phase25_start = time.time()
global_rg_cache = preextract_all_radgraph(all_candidates, batch_size=64)
phase25_time = time.time() - phase25_start
print(f"  ✅ Phase 2.5: {len(global_rg_cache)} texts, {phase25_time:.1f}s")




print("\n▶ PHASE 3: DUAL-TRACK Score + SAFE Fusion + Hedge + Correct (V3.5, vectorized)...")
phase3_start = time.time()

results = []
entity_score_cache = {}
failed_score = 0
n_constrained_blocked = 0
n_beam_reranked = 0
n_fused_reports = 0
n_fused_sentences_total = 0

for idx, sample in enumerate(tqdm(test_samples, desc="Phase 3: V3.5")):
    try:
        study_id = sample.get('id', f'test_{idx}')
        img_p = sample['image_path']
        if isinstance(img_p, list):
            img_p = img_p[0]
        image_path = str(IMAGE_DIR / img_p)
        candidates = all_candidates[idx]
        visual_features = visual_cache[image_path]

        
        candidate_texts = [c['report'] for c in candidates]
        all_candidate_entities = []
        for ct in candidate_texts:
            if ct in global_rg_cache:
                all_candidate_entities.append(global_rg_cache[ct])
            else:
                
                ents = extract_entities_batch_radgraph([ct])
                all_candidate_entities.append(ents[0] if ents else [])

        
        log_probs = [c['log_prob'] for c in candidates]
        min_lp, max_lp = min(log_probs), max(log_probs)
        lp_range = max_lp - min_lp if max_lp > min_lp else 1.0

        scored_candidates = []

        for cand, entities in zip(candidates, all_candidate_entities):
            
            
            
            
            
            
            obs_indices = []
            obs_texts = []
            obs_regions = []
            n_obs_da_skipped = 0
            for ei, ent in enumerate(entities):
                if "Anatomy" in ent.get("label", ""):
                    continue
                
                if ent.get("label", "") == "OBS-DA":
                    ent["score"] = None
                    ent["is_observation"] = True
                    ent["is_obs_da"] = True
                    ent["is_supported"] = True  
                    n_obs_da_skipped += 1
                    continue
                region_ids = []
                for rel in ent.get("relations", []):
                    if rel[0] == "located_at":
                        for other in entities:
                            if other.get("id") == rel[1] and "Anatomy" in other.get("label", ""):
                                region_ids.extend(map_anatomy_to_region(other.get("text", "")))
                if not region_ids:
                    region_ids = map_anatomy_to_region(ent.get("text", ""))
                obs_indices.append(ei)
                obs_texts.append(ent.get("text", ""))
                obs_regions.append(region_ids)
                ent["is_obs_da"] = False

            
            if obs_texts:
                scores = score_entities_batch(visual_features, obs_texts, obs_regions, entity_score_cache)
            else:
                scores = []

            score_idx = 0
            for ei, ent in enumerate(entities):
                if ei in obs_indices:
                    s = scores[score_idx]
                    ent["score"] = s
                    ent["is_observation"] = True
                    ent["is_supported"] = s >= 0.5
                    score_idx += 1
                else:
                    ent["score"] = None
                    ent["is_observation"] = False

            avg_ev = np.mean(scores) if scores else 0.5
            n_obs = len(scores)
            n_sup = sum(1 for s in scores if s >= 0.5)
            norm_lp = (cand['log_prob'] - min_lp) / lp_range if lp_range > 0 else 1.0

            
            combined = BEAM_ALPHA_EVIDENCE * avg_ev + BEAM_ALPHA_LOGPROB * norm_lp

            scored_candidates.append({
                **cand,
                'entities': entities,
                'n_entities': len(entities),
                'n_obs': n_obs,
                'n_supported': n_sup,
                'avg_evidence': avg_ev,
                'norm_logprob': norm_lp,
                'combined_score': combined,
            })

        
        beam_cands = [c for c in scored_candidates if c.get('strategy') == 'beam']
        nonbeam_cands = [c for c in scored_candidates if c.get('strategy') != 'beam']

        
        best_beam = max(beam_cands, key=lambda x: x['combined_score']) if beam_cands else scored_candidates[0]
        best = best_beam
        constrained_blocked = False

        
        beam_reranked = (best_beam['rank'] != 0) if beam_cands else False
        if beam_reranked:
            n_beam_reranked += 1

        
        if nonbeam_cands:
            best_nonbeam = max(nonbeam_cands, key=lambda x: x['combined_score'])
            if best_nonbeam['combined_score'] > best['combined_score'] + MIN_RERANK_GAIN:
                best = best_nonbeam
            elif best_nonbeam['combined_score'] > best['combined_score']:
                constrained_blocked = True
                n_constrained_blocked += 1

        baseline = next((c for c in scored_candidates if c['rank'] == 0), scored_candidates[0])
        

        
        fused_report, n_fused = sentence_fusion(
            best['report'], best['entities'], scored_candidates
        )
        if n_fused > 0:
            n_fused_reports += 1
            n_fused_sentences_total += n_fused

        
        hedged_report, n_hedged_this = apply_hedging(fused_report, best['entities'])

        
        corrected_report, n_removed = apply_conservative_correction(
            hedged_report, best['entities']
        )

        
        total_reverify_rounds = 0
        for reverify_t in range(MAX_REVERIFICATION):
            if corrected_report == best['report']:
                break  

            
            if corrected_report in global_rg_cache:
                _rev_entities = global_rg_cache[corrected_report]
            else:
                _rev_entities = extract_entities_radgraph(corrected_report)

            _rev_obs_texts = []
            _rev_obs_regions = []
            for ent in _rev_entities:
                if "Anatomy" in ent.get("label", ""):
                    continue
                
                if ent.get("label", "") == "OBS-DA":
                    ent["score"] = None
                    ent["is_observation"] = True
                    ent["is_obs_da"] = True
                    ent["is_supported"] = True
                    continue
                _rids = []
                for rel in ent.get("relations", []):
                    if rel[0] == "located_at":
                        for oth in _rev_entities:
                            if oth.get("id") == rel[1] and "Anatomy" in oth.get("label", ""):
                                _rids.extend(map_anatomy_to_region(oth.get("text", "")))
                if not _rids:
                    _rids = map_anatomy_to_region(ent.get("text", ""))
                ent["is_observation"] = True
                ent["is_obs_da"] = False
                _rev_obs_texts.append(ent.get("text", ""))
                _rev_obs_regions.append(_rids)
            if _rev_obs_texts:
                _rev_scores = score_entities_batch(
                    visual_features, _rev_obs_texts, _rev_obs_regions, entity_score_cache
                )
                _score_idx = 0
                for ent in _rev_entities:
                    if ent.get("is_observation") and not ent.get("is_obs_da"):
                        ent["score"] = _rev_scores[_score_idx]
                        ent["is_supported"] = _rev_scores[_score_idx] >= 0.5
                        _score_idx += 1
                _n_unsup = sum(1 for s in _rev_scores if s < REMOVE_THRESHOLD)
                _n_border = sum(1 for s in _rev_scores if REMOVE_THRESHOLD <= s < HEDGE_THRESHOLD)
                if _n_unsup == 0 and _n_border == 0:
                    break  
                _h_report, _h_n = apply_hedging(corrected_report, _rev_entities)
                _c_report, _c_n = apply_conservative_correction(_h_report, _rev_entities)
                if _c_report == corrected_report:
                    break  
                corrected_report = _c_report
                n_hedged_this += _h_n
                n_removed += _c_n
                total_reverify_rounds += 1
            else:
                break

        
        if corrected_report == best['report']:
            avg_score_corrected = best['avg_evidence']
            n_ent_corrected = best['n_entities']
            n_obs_corrected = best['n_obs']
            n_sup_corrected = best['n_supported']
        else:
            
            if corrected_report in global_rg_cache:
                corrected_entities = global_rg_cache[corrected_report]
            else:
                corrected_entities = extract_entities_radgraph(corrected_report)
            c_obs_texts = []
            c_obs_regions = []
            n_obs_da_final = 0
            for ent in corrected_entities:
                if "Anatomy" in ent.get("label", ""):
                    continue
                
                if ent.get("label", "") == "OBS-DA":
                    n_obs_da_final += 1
                    continue
                region_ids = []
                for rel in ent.get("relations", []):
                    if rel[0] == "located_at":
                        for other in corrected_entities:
                            if other.get("id") == rel[1] and "Anatomy" in other.get("label", ""):
                                region_ids.extend(map_anatomy_to_region(other.get("text", "")))
                if not region_ids:
                    region_ids = map_anatomy_to_region(ent.get("text", ""))
                c_obs_texts.append(ent.get("text", ""))
                c_obs_regions.append(region_ids)
            if c_obs_texts:
                c_scores = score_entities_batch(visual_features, c_obs_texts, c_obs_regions, entity_score_cache)
                avg_score_corrected = np.mean(c_scores)
                n_sup_corrected = sum(1 for s in c_scores if s >= 0.5)
            else:
                avg_score_corrected = 0.5
                n_sup_corrected = 0
            n_ent_corrected = len(corrected_entities)
            n_obs_corrected = len(c_obs_texts)

        results.append({
            "study_id": study_id,
            "ground_truth": sample.get('report', ''),
            "baseline_report": baseline['report'],
            "baseline_avg_evidence": baseline['avg_evidence'],
            "n_entities_baseline": baseline['n_entities'],
            "n_obs_baseline": baseline['n_obs'],
            "n_supported_baseline": baseline['n_supported'],
            "reranked_report": best['report'],
            "reranked_avg_evidence": best['avg_evidence'],
            "reranked_rank": best['rank'],
            "reranked_strategy": best.get('strategy', 'beam'),
            "reranked_beam_reranked": beam_reranked,
            "n_candidates": len(scored_candidates),
            "n_unique_candidates": len(candidates),
            "rerank_changed": best['rank'] != 0,
            "constrained_blocked": constrained_blocked,
            "n_sentences_fused": n_fused,
            "n_entities_hedged": n_hedged_this,
            "n_reverify_rounds": total_reverify_rounds,
            "corrected_report": corrected_report,
            "corrected_avg_evidence": float(avg_score_corrected),
            "n_entities_corrected": n_ent_corrected,
            "n_obs_corrected": n_obs_corrected,
            "n_supported_corrected": n_sup_corrected,
            "n_sentences_removed": n_removed,
            "evidence_improvement": float(avg_score_corrected) - baseline['avg_evidence'],
            "rerank_improvement": best['avg_evidence'] - baseline['avg_evidence'],
        })
    except Exception as e:
        failed_score += 1
        if failed_score <= 5:
            print(f"  ⚠ Failed: {sample.get('id', idx)}: {str(e)[:100]}")

phase3_time = time.time() - phase3_start
total_time = time.time() - total_start

print(f"\n  ✅ Phase 3: {len(results)} reports, {phase3_time:.1f}s ({phase3_time/60:.1f}min)")
if failed_score > 0:
    print(f"     Failed: {failed_score}")


if len(results) == 0:
    raise RuntimeError(
        f"Phase 3 produced 0 reports out of {n_test}. "
        f"All {failed_score} samples failed. Check error messages above."
    )


results_df = pd.DataFrame(results)
n_reranked = int(results_df['rerank_changed'].sum())
n_with_removal = int((results_df['n_sentences_removed'] > 0).sum())
n_with_fusion = int((results_df['n_sentences_fused'] > 0).sum())
n_hedged_total = int(results_df['n_entities_hedged'].sum())
n_reverified = int((results_df['n_reverify_rounds'] > 0).sum())
avg_unique = results_df['n_unique_candidates'].mean()
strategy_counts = results_df['reranked_strategy'].value_counts()

print(f"\n{'='*70}")
print(f"V3.5 PIPELINE COMPLETE — {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"{'='*70}")
print(f"\n⏱  Phase Timing:")
print(f"   Phase 1 (R2Gen FP32):          {phase1_time:>7.1f}s  ({phase1_time/total_time*100:4.1f}%)")
print(f"   Phase 2 (DenseNet DataLoader):  {phase2_time:>7.1f}s  ({phase2_time/total_time*100:4.1f}%)")
print(f"   Phase 2.5 (RadGraph global):    {phase25_time:>7.1f}s  ({phase25_time/total_time*100:4.1f}%)")
print(f"   Phase 3 (V3.5 vectorized):      {phase3_time:>7.1f}s  ({phase3_time/total_time*100:4.1f}%)")
print(f"   Speed: {total_time/max(len(results),1)*1000:.0f}ms/report")

print(f"\n📊 V3.5 Stats:")
print(f"   Reports: {len(results)}/{n_test}")
print(f"   Re-ranked (total): {n_reranked}/{len(results)} ({n_reranked/max(len(results),1)*100:.1f}%)")
print(f"   ├── Beam re-ranked (beam-1/2 beat beam-0): {n_beam_reranked}")
print(f"   ├── Non-beam winners: {n_reranked - n_beam_reranked}")
print(f"   └── Non-beam blocked (gain < {MIN_RERANK_GAIN}): {n_constrained_blocked}")
print(f"   Sentence fusion: {n_fused_reports} reports, {n_fused_sentences_total} sentences fused")
print(f"   Hedging: {n_hedged_total} entities hedged")
print(f"   Re-verification: {n_reverified} reports needed ≥1 extra pass")
print(f"   Sentences removed: {int(results_df['n_sentences_removed'].sum())} (from {n_with_removal} reports)")
print(f"   Avg candidates: {avg_unique:.2f}")
print(f"   Entity cache: {len(entity_score_cache)} entries")
print(f"   RadGraph cache: {len(global_rg_cache)} entries")

print(f"\n📊 Strategy Distribution:")
for strat, count in strategy_counts.items():
    print(f"   {strat}: {count} ({count/len(results)*100:.1f}%)")

print(f"   Entity cache: {len(entity_score_cache)} entries")
print(f"   RadGraph cache: {len(global_rg_cache)} entries")


print(f"\n📊 Strategy Distribution:")
for strat, count in strategy_counts.items():
    print(f"   {strat}: {count} ({count/len(results)*100:.1f}%)")


print(f"\n📊 Evidence Preview:")
preview_cols = [
    'study_id', 
    'baseline_avg_evidence', 
    'reranked_avg_evidence',
    'corrected_avg_evidence', 
    'evidence_improvement'
]
print(results_df[preview_cols].head(10))

In [ ]:
!pip install pycocoevalcap











from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
try:
    from pycocoevalcap.meteor.meteor import Meteor
    _HAS_METEOR = True
except Exception:
    _HAS_METEOR = False

from radgraph import F1RadGraph


def clean_report_text(text: str) -> str:
    
    text = text.lower().strip()
    text = re.sub(r'\bxxxx\b', '', text)
    
    text = re.sub(r'([a-z])\.([a-z])', r'\1 . \2', text)
    text = re.sub(r'([a-z]),([a-z])', r'\1 , \2', text)
    text = re.sub(r'([a-z]):([a-z])', r'\1 : \2', text)
    
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def compute_nlg_metrics(references: List[str], hypotheses: List[str]) -> Dict:
    gts = {i: [clean_report_text(ref)] for i, ref in enumerate(references)}
    res = {i: [clean_report_text(hyp)] for i, hyp in enumerate(hypotheses)}
    bleu_scorer = Bleu(4)
    bleu_scores, _ = bleu_scorer.compute_score(gts, res, verbose=0)
    rouge_obj = Rouge()
    rouge_score, _ = rouge_obj.compute_score(gts, res)
    cider_scorer = Cider()
    cider_score, _ = cider_scorer.compute_score(gts, res)
    if _HAS_METEOR:
        try:
            meteor_scorer = Meteor()
            meteor_score, _ = meteor_scorer.compute_score(gts, res)
        except Exception:
            meteor_score = 0.0
    else:
        meteor_score = 0.0
    return {
        'BLEU-1': bleu_scores[0], 'BLEU-2': bleu_scores[1],
        'BLEU-3': bleu_scores[2], 'BLEU-4': bleu_scores[3],
        'ROUGE-L': rouge_score, 'CIDEr-D': cider_score, 'METEOR': meteor_score,
    }

print("Computing NLG metrics (3-way, V3.5 balanced preprocessing)...")
gt_texts = [r['ground_truth'] for r in results]
baseline_nlg  = compute_nlg_metrics(gt_texts, [r['baseline_report'] for r in results])
reranked_nlg  = compute_nlg_metrics(gt_texts, [r['reranked_report'] for r in results])
corrected_nlg = compute_nlg_metrics(gt_texts, [r['corrected_report'] for r in results])
print("✅ NLG metrics computed (3-way + METEOR)")


print("\nComputing RadGraph F1 (3-way)...")
f1_radgraph = F1RadGraph(reward_level="all", model_type="radgraph-xl", cuda=0)

baseline_hyps = [r['baseline_report'] for r in results]
rg_baseline = f1_radgraph(hyps=baseline_hyps, refs=gt_texts)
rg_baseline_scores = {'RG_E': rg_baseline[0][0], 'RG_ER': rg_baseline[0][1], 'RG_bar_ER': rg_baseline[0][2]}

reranked_hyps = [r['reranked_report'] for r in results]
rg_reranked = f1_radgraph(hyps=reranked_hyps, refs=gt_texts)
rg_reranked_scores = {'RG_E': rg_reranked[0][0], 'RG_ER': rg_reranked[0][1], 'RG_bar_ER': rg_reranked[0][2]}

corrected_hyps = [r['corrected_report'] for r in results]
rg_corrected = f1_radgraph(hyps=corrected_hyps, refs=gt_texts)
rg_corrected_scores = {'RG_E': rg_corrected[0][0], 'RG_ER': rg_corrected[0][1], 'RG_bar_ER': rg_corrected[0][2]}
print("✅ RadGraph F1 computed (3-way)")


print("\nComputing safety metrics...")
avg_ev_b = np.mean([r['baseline_avg_evidence'] for r in results])
avg_ev_r = np.mean([r['reranked_avg_evidence'] for r in results])
avg_ev_c = np.mean([r['corrected_avg_evidence'] for r in results])
sr_b = np.mean([r['n_supported_baseline'] / max(r['n_obs_baseline'], 1) for r in results])
sr_c = np.mean([r['n_supported_corrected'] / max(r['n_obs_corrected'], 1) for r in results])

safety_metrics = {
    'avg_evidence_baseline': float(avg_ev_b),
    'avg_evidence_reranked': float(avg_ev_r),
    'avg_evidence_corrected': float(avg_ev_c),
    'evidence_improvement': float(avg_ev_c - avg_ev_b),
    'rerank_improvement': float(avg_ev_r - avg_ev_b),
    'support_rate_baseline': float(sr_b),
    'support_rate_corrected': float(sr_c),
    'support_rate_improvement': float(sr_c - sr_b),
}
print("✅ Safety metrics computed")


print("\n" + "=" * 70)
print("RESULTS: BASELINE → RE-RANKED → CORRECTED (V3.5)")
print("=" * 70)

print("\n📊 NLG METRICS:")
print(f"{'Metric':<10} {'Baseline':>10} {'Reranked':>10} {'Corrected':>10} {'Δ(B→R)':>10} {'Δ(B→C)':>10}")
print("-" * 62)
for key in ['BLEU-1','BLEU-2','BLEU-3','BLEU-4','ROUGE-L','CIDEr-D','METEOR']:
    b, r, c = baseline_nlg[key], reranked_nlg[key], corrected_nlg[key]
    print(f"{key:<10} {b:>10.4f} {r:>10.4f} {c:>10.4f} {r-b:>+10.4f} {c-b:>+10.4f}")

print("\n📊 RADGRAPH F1:")
print(f"{'Metric':<10} {'Baseline':>10} {'Reranked':>10} {'Corrected':>10} {'Δ(B→R)':>10} {'Δ(B→C)':>10}")
print("-" * 62)
for key in ['RG_E','RG_ER','RG_bar_ER']:
    b, r, c = rg_baseline_scores[key], rg_reranked_scores[key], rg_corrected_scores[key]
    print(f"{key:<10} {b:>10.4f} {r:>10.4f} {c:>10.4f} {r-b:>+10.4f} {c-b:>+10.4f}")

print("\n📊 SAFETY / EVIDENCE:")
print(f"{'Metric':<28} {'Baseline':>10} {'Reranked':>10} {'Corrected':>10} {'Δ(B→C)':>10}")
print("-" * 70)
print(f"{'Avg Evidence':<28} {avg_ev_b:>10.4f} {avg_ev_r:>10.4f} {avg_ev_c:>10.4f} {avg_ev_c-avg_ev_b:>+10.4f}")
print(f"{'Support Rate':<28} {sr_b:>10.2%} {'—':>10} {sr_c:>10.2%} {sr_c-sr_b:>+10.2%}")


n_blocked = sum(1 for r in results if r.get('constrained_blocked', False))
n_rr = int(results_df['rerank_changed'].sum())
n_beam_rr = sum(1 for r in results if r.get('reranked_beam_reranked', False))
n_fused_r = sum(1 for r in results if r.get('n_sentences_fused', 0) > 0)
n_fused_s = sum(r.get('n_sentences_fused', 0) for r in results)
n_hedged_r = sum(1 for r in results if r.get('n_entities_hedged', 0) > 0)
n_hedged_s = sum(r.get('n_entities_hedged', 0) for r in results)
n_reverified_r = sum(1 for r in results if r.get('n_reverify_rounds', 0) > 0)
print(f"\n📊 V3.5 ABLATION — Dual-Track + Safe Fusion + Hedging + Re-verify:")
print(f"   Re-ranked total: {n_rr}/{len(results)} ({n_rr/len(results)*100:.1f}%)")
print(f"   ├── Beam re-ranked (beam-1/2 beat beam-0): {n_beam_rr}")
print(f"   ├── Non-beam winners: {n_rr - n_beam_rr}")
print(f"   └── Non-beam blocked: {n_blocked}")
print(f"   Sentence fusion: {n_fused_r} reports, {n_fused_s} sentences fused")
print(f"   Hedging: {n_hedged_r} reports, {n_hedged_s} entities hedged")
print(f"   Re-verification: {n_reverified_r} reports needed ≥1 extra pass")
print(f"   Sentences removed: {int(results_df['n_sentences_removed'].sum())} (from {int((results_df['n_sentences_removed']>0).sum())} reports)")


def compute_scas(rl):
    ts_b = sum(r['n_supported_baseline'] for r in rl)
    to_b = sum(r['n_obs_baseline'] for r in rl)
    ts_c = sum(r['n_supported_corrected'] for r in rl)
    to_c = sum(r['n_obs_corrected'] for r in rl)
    sb = ts_b / max(to_b, 1)
    sc = ts_c / max(to_c, 1)
    normal = [r for r in rl if r['n_obs_baseline'] <= 6]
    abnormal = [r for r in rl if r['n_obs_baseline'] > 6]
    def _s(sub, ks, ko):
        return sum(r[ks] for r in sub) / max(sum(r[ko] for r in sub), 1)
    return {
        'scas_baseline': sb, 'scas_corrected': sc, 'scas_delta': sc - sb,
        'scas_normal_baseline': _s(normal, 'n_supported_baseline', 'n_obs_baseline'),
        'scas_normal_corrected': _s(normal, 'n_supported_corrected', 'n_obs_corrected'),
        'scas_abnormal_baseline': _s(abnormal, 'n_supported_baseline', 'n_obs_baseline'),
        'scas_abnormal_corrected': _s(abnormal, 'n_supported_corrected', 'n_obs_corrected'),
        'n_normal': len(normal), 'n_abnormal': len(abnormal),
        'total_obs_baseline': to_b, 'total_obs_corrected': to_c,
    }

scas = compute_scas(results)
print(f"\n📊 SCAS:")
print(f"  Overall:  {scas['scas_baseline']:.4f} → {scas['scas_corrected']:.4f} (Δ={scas['scas_delta']:+.4f})")
print(f"  Normal  (n={scas['n_normal']:>3}): {scas['scas_normal_baseline']:.4f} → {scas['scas_normal_corrected']:.4f}")
print(f"  Abnormal(n={scas['n_abnormal']:>3}): {scas['scas_abnormal_baseline']:.4f} → {scas['scas_abnormal_corrected']:.4f}")
print(f"  Obs entities: {scas['total_obs_baseline']} → {scas['total_obs_corrected']}")
print("✅ SCAS computed")


from scipy import stats

print("\n" + "=" * 70)
print("📊 STATISTICAL SIGNIFICANCE (V3.5)")
print("=" * 70)

baseline_ev = [r['baseline_avg_evidence'] for r in results]
corrected_ev = [r['corrected_avg_evidence'] for r in results]

t_stat, p_ttest = stats.ttest_rel(corrected_ev, baseline_ev)
print(f"\n  Paired t-test: t={t_stat:.4f}, p={p_ttest:.6f} {'✅' if p_ttest < 0.05 else '❌'}")

diffs_ev = [c - b for c, b in zip(corrected_ev, baseline_ev) if abs(c - b) > 1e-10]
if len(diffs_ev) >= 10:
    w_stat, p_wilcox = stats.wilcoxon(diffs_ev)
    print(f"  Wilcoxon ({len(diffs_ev)} pairs): W={w_stat:.1f}, p={p_wilcox:.6f} {'✅' if p_wilcox < 0.05 else '❌'}")
else:
    w_stat, p_wilcox = 0.0, 1.0
    print(f"  Wilcoxon: too few non-tied pairs ({len(diffs_ev)})")

n_bootstrap = 10000
all_diffs = [c - b for c, b in zip(corrected_ev, baseline_ev)]
np.random.seed(SEED)
boot_means = [np.mean(np.random.choice(all_diffs, size=len(all_diffs), replace=True)) for _ in range(n_bootstrap)]
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
mean_diff = np.mean(all_diffs)
print(f"  Bootstrap CI: mean Δ={mean_diff:.4f}, 95% CI=[{ci_low:.4f}, {ci_high:.4f}] {'✅' if ci_low > 0 else '❌'}")

pooled_std = np.sqrt((np.std(baseline_ev)**2 + np.std(corrected_ev)**2) / 2)
cohens_d = mean_diff / pooled_std if pooled_std > 0 else 0
effect_label = "negligible" if abs(cohens_d) < 0.2 else "small" if abs(cohens_d) < 0.5 else "medium" if abs(cohens_d) < 0.8 else "large"
print(f"  Cohen's d: {cohens_d:.4f} ({effect_label})")

baseline_majority = [r['n_supported_baseline'] / max(r['n_obs_baseline'], 1) >= 0.5 for r in results]
corrected_majority = [r['n_supported_corrected'] / max(r['n_obs_corrected'], 1) >= 0.5 for r in results]
b_yes_c_no = sum(1 for b, c in zip(baseline_majority, corrected_majority) if b and not c)
b_no_c_yes = sum(1 for b, c in zip(baseline_majority, corrected_majority) if not b and c)
discordant = b_yes_c_no + b_no_c_yes
if discordant > 0:
    mcnemar_chi2 = (abs(b_yes_c_no - b_no_c_yes) - 1)**2 / discordant
    mcnemar_p = 1 - stats.chi2.cdf(mcnemar_chi2, df=1)
    print(f"  McNemar's: B+/C−={b_yes_c_no}, B−/C+={b_no_c_yes}, χ²={mcnemar_chi2:.3f}, p={mcnemar_p:.6f} {'✅' if mcnemar_p < 0.05 else '❌'}")
else:
    mcnemar_chi2, mcnemar_p = 0.0, 1.0


bleu4_d = corrected_nlg['BLEU-4'] - baseline_nlg['BLEU-4']
cider_d = corrected_nlg['CIDEr-D'] - baseline_nlg['CIDEr-D']
rouge_d = corrected_nlg['ROUGE-L'] - baseline_nlg['ROUGE-L']
rger_d = rg_corrected_scores['RG_ER'] - rg_baseline_scores['RG_ER']
meteor_d = corrected_nlg['METEOR'] - baseline_nlg['METEOR']
print(f"\n  NLG Degradation:")
print(f"    BLEU-4  Δ={bleu4_d:+.4f} ({bleu4_d/max(baseline_nlg['BLEU-4'],1e-6)*100:+.1f}%)")
print(f"    CIDEr-D Δ={cider_d:+.4f} ({cider_d/max(baseline_nlg['CIDEr-D'],1e-6)*100:+.1f}%)")
print(f"    ROUGE-L Δ={rouge_d:+.4f} ({rouge_d/max(baseline_nlg['ROUGE-L'],1e-6)*100:+.1f}%)")
print(f"    METEOR  Δ={meteor_d:+.4f} ({meteor_d/max(baseline_nlg['METEOR'],1e-6)*100:+.1f}%)")
print(f"    RG_ER   Δ={rger_d:+.4f} ({rger_d/max(rg_baseline_scores['RG_ER'],1e-6)*100:+.1f}%)")

significance = {
    'ttest_t': float(t_stat), 'ttest_p': float(p_ttest),
    'wilcoxon_w': float(w_stat), 'wilcoxon_p': float(p_wilcox),
    'bootstrap_mean_delta': float(mean_diff),
    'bootstrap_ci_low': float(ci_low), 'bootstrap_ci_high': float(ci_high),
    'cohens_d': float(cohens_d), 'effect_size': effect_label,
    'mcnemar_chi2': float(mcnemar_chi2), 'mcnemar_p': float(mcnemar_p),
    'n_non_tied_pairs': len(diffs_ev), 'n_bootstrap': n_bootstrap,
}
print("✅ All significance tests completed")


scas_f1_baseline = compute_scas_f1([{**r, 'n_supported_corrected': r['n_supported_baseline'],
                                      'n_obs_corrected': r['n_obs_baseline']} for r in results])
scas_f1_corrected = compute_scas_f1(results)
print(f"\n📊 SCAS-F1 (deletion-resistant):")
print(f"   Baseline: {scas_f1_baseline:.4f}")
print(f"   Corrected: {scas_f1_corrected:.4f}")
print(f"   Δ: {scas_f1_corrected - scas_f1_baseline:+.4f}")

In [ ]:
try:
    from chexbert.label import label as chexbert_label
    _HAS_CHEXBERT = True
    print("✅ CheXbert loaded")
except ImportError:
    try:
        
        from transformers import AutoModelForSequenceClassification, AutoTokenizer as _AT
        _chexbert_model = AutoModelForSequenceClassification.from_pretrained(
            "StanfordAIMI/CheXbert", trust_remote_code=True
        ).to(DEVICE).eval()
        _chexbert_tokenizer = _AT.from_pretrained("StanfordAIMI/CheXbert", trust_remote_code=True)
        _HAS_CHEXBERT = True
        print("✅ CheXbert loaded (transformers)")
    except Exception as e:
        _HAS_CHEXBERT = False
        print(f"⚠️ CheXbert not available: {e}")
        print("   Install with: pip install chexbert")
        print("   Or: pip install transformers && download StanfordAIMI/CheXbert")

if _HAS_CHEXBERT:
    from sklearn.metrics import f1_score as _f1_score, classification_report as _cls_report

    def compute_chexbert_f1(references, hypotheses):
        
        
        ref_labels = chexbert_label(references)   
        hyp_labels = chexbert_label(hypotheses)    

        
        ref_bin = (np.array(ref_labels) == 1).astype(int)
        hyp_bin = (np.array(hyp_labels) == 1).astype(int)

        PATHOLOGY_NAMES = [
            'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
            'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia',
            'Atelectasis', 'Pneumothorax', 'Pleural Effusion',
            'Pleural Other', 'Fracture', 'Support Devices', 'No Finding'
        ]

        
        per_class_f1 = []
        for i in range(14):
            f1 = _f1_score(ref_bin[:, i], hyp_bin[:, i], average='binary', zero_division=0)
            per_class_f1.append(f1)

        micro_f1 = _f1_score(ref_bin.flatten(), hyp_bin.flatten(), average='micro', zero_division=0)
        macro_f1 = np.mean(per_class_f1)

        
        CE_NAMES = ['Cardiomegaly', 'Edema', 'Consolidation', 'Atelectasis', 'Pleural Effusion']
        CE_INDICES = [PATHOLOGY_NAMES.index(n) for n in CE_NAMES]
        ce_f1 = np.mean([per_class_f1[i] for i in CE_INDICES])

        return {
            'micro_f1': float(micro_f1),
            'macro_f1': float(macro_f1),
            'ce5_f1': float(ce_f1),
            'per_class': {PATHOLOGY_NAMES[i]: float(per_class_f1[i]) for i in range(14)},
        }

    gt_texts = [r['ground_truth'] for r in results]
    print("Computing CheXbert F1 (baseline)...")
    chexbert_baseline = compute_chexbert_f1(gt_texts, [r['baseline_report'] for r in results])
    print("Computing CheXbert F1 (corrected)...")
    chexbert_corrected = compute_chexbert_f1(gt_texts, [r['corrected_report'] for r in results])

    print(f"\n{'='*70}")
    print(f"CheXbert F1 — INDEPENDENT Clinical Evaluation")
    print(f"{'='*70}")
    print(f"{'Metric':<20} {'Baseline':>10} {'Corrected':>10} {'Δ':>10}")
    print("-" * 52)
    print(f"{'Micro-F1':<20} {chexbert_baseline['micro_f1']:>10.4f} {chexbert_corrected['micro_f1']:>10.4f} {chexbert_corrected['micro_f1']-chexbert_baseline['micro_f1']:>+10.4f}")
    print(f"{'Macro-F1':<20} {chexbert_baseline['macro_f1']:>10.4f} {chexbert_corrected['macro_f1']:>10.4f} {chexbert_corrected['macro_f1']-chexbert_baseline['macro_f1']:>+10.4f}")
    print(f"{'CE-5 F1':<20} {chexbert_baseline['ce5_f1']:>10.4f} {chexbert_corrected['ce5_f1']:>10.4f} {chexbert_corrected['ce5_f1']-chexbert_baseline['ce5_f1']:>+10.4f}")

    print(f"\nPer-class F1:")
    for name in chexbert_baseline['per_class']:
        b = chexbert_baseline['per_class'][name]
        c = chexbert_corrected['per_class'][name]
        marker = "↑" if c > b else "↓" if c < b else "="
        print(f"   {name:<30} {b:.4f} → {c:.4f} ({c-b:+.4f}) {marker}")
else:
    print("\n⚠️ Skipping CheXbert evaluation — install CheXbert for independent clinical metrics")
    print("   This is REQUIRED before paper submission (breaks circular evaluation)")
    chexbert_baseline = None
    chexbert_corrected = None

In [ ]:
import sys, os
sys.path.insert(0, '/kaggle/working')
try:
    from src.evaluation_utils import (
        green_score_reports, green_score_available,
        failure_taxonomy, tradeoff_analysis,
    )
    _HAS_EVAL_UTILS = True
except ImportError:
    _HAS_EVAL_UTILS = False


print("=" * 70)
print("GREEN EVALUATION — Independent Report Quality (LLM-based)")
print("=" * 70)

gt_texts = [r['ground_truth'] for r in results]
baseline_texts = [r['baseline_report'] for r in results]
corrected_texts = [r['corrected_report'] for r in results]

green_baseline = None
green_corrected = None

if _HAS_EVAL_UTILS and green_score_available():
    print("Computing GREEN scores (baseline)...")
    green_baseline = green_score_reports(
        generated=baseline_texts, reference=gt_texts,
        device=str(DEVICE), batch_size=4,
    )
    print("Computing GREEN scores (corrected)...")
    green_corrected = green_score_reports(
        generated=corrected_texts, reference=gt_texts,
        device=str(DEVICE), batch_size=4,
    )

    if green_baseline.get("available") and green_corrected.get("available"):
        print(f"\n  GREEN — Baseline:  {green_baseline['mean_green']:.4f} ± {green_baseline.get('std_green', 0):.4f}")
        print(f"  GREEN — Corrected: {green_corrected['mean_green']:.4f} ± {green_corrected.get('std_green', 0):.4f}")
        delta_g = green_corrected['mean_green'] - green_baseline['mean_green']
        print(f"  GREEN — Δ:         {delta_g:+.4f}")
        print("✅ GREEN evaluation complete")
    else:
        err_b = green_baseline.get("error", "unknown")
        err_c = green_corrected.get("error", "unknown")
        print(f"⚠️ GREEN evaluation failed:")
        print(f"   Baseline: {err_b}")
        print(f"   Corrected: {err_c}")
else:
    print("⚠️ GREEN evaluation not available.")
    if not _HAS_EVAL_UTILS:
        print("   src.evaluation_utils not importable")
    else:
        print("   green_score package not installed. Install: pip install green-score")
    print("   Continuing without GREEN — document this limitation in the paper.")


green_summary = {
    'baseline': green_baseline if green_baseline else {'available': False},
    'corrected': green_corrected if green_corrected else {'available': False},
}
green_path = OUTPUT_DIR / 'green_summary.json'
with open(green_path, 'w') as f:
    json.dump(green_summary, f, indent=2, default=str)
print(f"✅ Saved: {green_path}")


print("\n" + "=" * 70)
print("FAILURE TAXONOMY — Correction Outcome Classification")
print("=" * 70)



failure_rows = []
for r in results:
    ev_baseline = r['baseline_avg_evidence']
    ev_corrected = r['corrected_avg_evidence']
    ev_delta = ev_corrected - ev_baseline
    was_modified = (r['corrected_report'] != r['baseline_report'])
    n_removed = r['n_sentences_removed']
    n_hedged = r.get('n_entities_hedged', 0)
    n_fused = r.get('n_sentences_fused', 0)

    
    
    
    
    
    

    if was_modified and ev_delta > 0.01:
        category = "true_positive_correction"
    elif was_modified and ev_delta < -0.01:
        category = "overcorrection"
    elif was_modified and abs(ev_delta) <= 0.01:
        category = "neutral_modification"
    elif not was_modified and ev_baseline >= 0.55:
        category = "no_change_needed"
    elif not was_modified and ev_baseline < 0.55:
        category = "missed_opportunity"
    else:
        category = "other"

    failure_rows.append({
        'study_id': r['study_id'],
        'category': category,
        'baseline_evidence': ev_baseline,
        'corrected_evidence': ev_corrected,
        'evidence_delta': ev_delta,
        'was_modified': was_modified,
        'n_removed': n_removed,
        'n_hedged': n_hedged,
        'n_fused': n_fused,
    })

failure_df = pd.DataFrame(failure_rows)
category_counts = failure_df['category'].value_counts()

print("\nCorrection Outcome Distribution:")
for cat, count in category_counts.items():
    pct = count / len(failure_df) * 100
    print(f"  {cat:<30} {count:>4} ({pct:>5.1f}%)")


print("\nPer-Category Evidence Statistics:")
print(f"  {'Category':<30} {'N':>4} {'Baseline':>9} {'Corrected':>9} {'Δ':>8}")
print("  " + "-" * 64)
for cat in category_counts.index:
    sub = failure_df[failure_df['category'] == cat]
    print(f"  {cat:<30} {len(sub):>4} {sub['baseline_evidence'].mean():>9.4f} "
          f"{sub['corrected_evidence'].mean():>9.4f} {sub['evidence_delta'].mean():>+8.4f}")


failure_path = OUTPUT_DIR / 'failure_analysis.csv'
failure_df.to_csv(failure_path, index=False)
print(f"\n✅ Saved: {failure_path}")


print("\n" + "=" * 70)
print("TRADE-OFF ANALYSIS — Fluency vs Clinical Accuracy")
print("=" * 70)


tradeoff_rows = []


for metric in ['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4', 'ROUGE-L', 'CIDEr-D', 'METEOR']:
    b_val = baseline_nlg.get(metric, 0)
    c_val = corrected_nlg.get(metric, 0)
    delta = c_val - b_val
    pct_chg = (delta / abs(b_val) * 100) if b_val != 0 else 0
    direction = "improved" if delta > 0.0005 else ("degraded" if delta < -0.0005 else "unchanged")
    tradeoff_rows.append({
        'metric': metric, 'category': 'NLG (fluency)',
        'baseline': round(b_val, 4), 'corrected': round(c_val, 4),
        'delta': round(delta, 4), 'pct_change': round(pct_chg, 2),
        'direction': direction,
    })


for metric in ['RG_E', 'RG_ER', 'RG_bar_ER']:
    b_val = rg_baseline_scores.get(metric, 0)
    c_val = rg_corrected_scores.get(metric, 0)
    delta = c_val - b_val
    pct_chg = (delta / abs(b_val) * 100) if b_val != 0 else 0
    direction = "improved" if delta > 0.0005 else ("degraded" if delta < -0.0005 else "unchanged")
    tradeoff_rows.append({
        'metric': metric, 'category': 'RadGraph (clinical)',
        'baseline': round(b_val, 4), 'corrected': round(c_val, 4),
        'delta': round(delta, 4), 'pct_change': round(pct_chg, 2),
        'direction': direction,
    })


tradeoff_rows.append({
    'metric': 'Avg Evidence', 'category': 'Evidence (self-derived)',
    'baseline': round(float(avg_ev_b), 4), 'corrected': round(float(avg_ev_c), 4),
    'delta': round(float(avg_ev_c - avg_ev_b), 4),
    'pct_change': round(float((avg_ev_c - avg_ev_b) / avg_ev_b * 100), 2),
    'direction': 'improved' if avg_ev_c > avg_ev_b else 'degraded',
})
tradeoff_rows.append({
    'metric': 'Support Rate', 'category': 'Evidence (self-derived)',
    'baseline': round(float(sr_b), 4), 'corrected': round(float(sr_c), 4),
    'delta': round(float(sr_c - sr_b), 4),
    'pct_change': round(float((sr_c - sr_b) / max(sr_b, 1e-6) * 100), 2),
    'direction': 'improved' if sr_c > sr_b else 'degraded',
})
tradeoff_rows.append({
    'metric': 'SCAS', 'category': 'Evidence (self-derived)',
    'baseline': round(scas['scas_baseline'], 4),
    'corrected': round(scas['scas_corrected'], 4),
    'delta': round(scas['scas_delta'], 4),
    'pct_change': round(scas['scas_delta'] / max(scas['scas_baseline'], 1e-6) * 100, 2),
    'direction': 'improved' if scas['scas_delta'] > 0 else 'degraded',
})


if green_baseline and green_baseline.get('available') and green_corrected and green_corrected.get('available'):
    g_b = green_baseline['mean_green']
    g_c = green_corrected['mean_green']
    g_d = g_c - g_b
    tradeoff_rows.append({
        'metric': 'GREEN', 'category': 'Independent (LLM)',
        'baseline': round(g_b, 4), 'corrected': round(g_c, 4),
        'delta': round(g_d, 4),
        'pct_change': round(g_d / max(abs(g_b), 1e-6) * 100, 2),
        'direction': 'improved' if g_d > 0.0005 else ('degraded' if g_d < -0.0005 else 'unchanged'),
    })


if chexbert_baseline is not None and chexbert_corrected is not None:
    for cb_metric in ['micro_f1', 'macro_f1', 'ce5_f1']:
        b_val = chexbert_baseline[cb_metric]
        c_val = chexbert_corrected[cb_metric]
        delta = c_val - b_val
        pct_chg = (delta / abs(b_val) * 100) if b_val != 0 else 0
        tradeoff_rows.append({
            'metric': f'CheXbert {cb_metric}', 'category': 'Independent (CheXbert)',
            'baseline': round(b_val, 4), 'corrected': round(c_val, 4),
            'delta': round(delta, 4), 'pct_change': round(pct_chg, 2),
            'direction': 'improved' if delta > 0.0005 else ('degraded' if delta < -0.0005 else 'unchanged'),
        })

tradeoff_df = pd.DataFrame(tradeoff_rows)


print(f"\n{'Metric':<25} {'Category':<25} {'Baseline':>9} {'Corrected':>9} {'Δ':>8} {'%':>7} {'':>10}")
print("-" * 95)
for _, row in tradeoff_df.iterrows():
    marker = '↑' if row['direction'] == 'improved' else ('↓' if row['direction'] == 'degraded' else '=')
    print(f"  {row['metric']:<23} {row['category']:<25} {row['baseline']:>9.4f} {row['corrected']:>9.4f} "
          f"{row['delta']:>+8.4f} {row['pct_change']:>+6.1f}% {marker}")


n_improved = (tradeoff_df['direction'] == 'improved').sum()
n_degraded = (tradeoff_df['direction'] == 'degraded').sum()
n_unchanged = (tradeoff_df['direction'] == 'unchanged').sum()
print(f"\nSummary: {n_improved} improved, {n_degraded} degraded, {n_unchanged} unchanged")


n_obs_baseline = sum(r['n_obs_baseline'] for r in results)
n_obs_corrected = sum(r['n_obs_corrected'] for r in results)
obs_reduction = n_obs_baseline - n_obs_corrected
obs_reduction_pct = obs_reduction / max(n_obs_baseline, 1) * 100

print(f"\n⚠️  HONEST TRADE-OFF NOTE (for paper):")
print(f"   Total observations: {n_obs_baseline} → {n_obs_corrected} (Δ={-obs_reduction}, {-obs_reduction_pct:.1f}%)")
print(f"   The pipeline removes {obs_reduction} observation entities ({obs_reduction_pct:.1f}% of total).")
print(f"   This means some CORRECT findings may also be removed (recall trade-off).")
print(f"   Paper must explicitly state this limitation per CLAIM 2024 item 


tradeoff_path = OUTPUT_DIR / 'tradeoff_analysis.csv'
tradeoff_df.to_csv(tradeoff_path, index=False)
print(f"\n✅ Saved: {tradeoff_path}")

print("\n✅ Cell 8c complete — GREEN + Failure Taxonomy + Trade-off Analysis")

In [ ]:
print("=" * 70)
print("QUALITY CHECKS (V3.5)")
print("=" * 70)

bleu4_drop = corrected_nlg['BLEU-4'] - baseline_nlg['BLEU-4']
rouge_drop = corrected_nlg['ROUGE-L'] - baseline_nlg['ROUGE-L']
cider_drop = corrected_nlg['CIDEr-D'] - baseline_nlg['CIDEr-D']
rger_drop  = rg_corrected_scores['RG_ER'] - rg_baseline_scores['RG_ER']

checks = [
    (len(results) >= 200, f"Processed {len(results)}/N (≥200)"),
    (safety_metrics['evidence_improvement'] >= 0, f"Evidence ↑: {safety_metrics['evidence_improvement']:.4f}"),
    (safety_metrics['support_rate_corrected'] >= safety_metrics['support_rate_baseline'],
     f"Support rate: {safety_metrics['support_rate_baseline']:.2%} → {safety_metrics['support_rate_corrected']:.2%}"),
    (rg_baseline_scores['RG_ER'] > 0.2, f"Baseline RG_ER > 0.2: {rg_baseline_scores['RG_ER']:.4f}"),
    (baseline_nlg['BLEU-4'] > 0.10, f"Baseline BLEU-4 > 0.10: {baseline_nlg['BLEU-4']:.4f}"),
    (abs(bleu4_drop) < 0.010, f"BLEU-4 Δ < 1.0%: {bleu4_drop:+.4f} ({'PASS' if abs(bleu4_drop) < 0.010 else 'FAIL'})"),
    (abs(rouge_drop) < 0.010, f"ROUGE-L Δ < 1.0%: {rouge_drop:+.4f} ({'PASS' if abs(rouge_drop) < 0.010 else 'FAIL'})"),
    (abs(cider_drop) < 0.050, f"CIDEr-D Δ < 5.0%: {cider_drop:+.4f} ({'PASS' if abs(cider_drop) < 0.050 else 'FAIL'})"),
    (abs(rger_drop) < 0.010, f"RG_ER Δ < 1.0%: {rger_drop:+.4f} ({'PASS' if abs(rger_drop) < 0.010 else 'FAIL'})"),
    (significance['ttest_p'] < 0.05, f"Evidence p={significance['ttest_p']:.6f}"),
]

all_passed = True
for passed, msg in checks:
    status = "✅" if passed else "⚠️"
    print(f"{status} {msg}")
    if not passed:
        all_passed = False

print(f"\n{'🎯 ALL CHECKS PASSED' if all_passed else '⚠️ Some checks need attention'}")


print("\n" + "=" * 70)
print("SAMPLE CORRECTIONS (V3.5)")
print("=" * 70)

by_imp = sorted(results, key=lambda x: x['evidence_improvement'], reverse=True)
print(f"\nReports with evidence Δ > 0.05: {sum(1 for r in results if r['evidence_improvement'] > 0.05)}")

for i, r in enumerate(by_imp[:5]):
    print(f"\n[{i+1}] {r['study_id']}")
    print(f"    Evidence: {r['baseline_avg_evidence']:.3f} → {r['corrected_avg_evidence']:.3f} ({r['evidence_improvement']:+.3f})")
    print(f"    Strategy: {r.get('reranked_strategy','beam')} | Beam-reranked: {r.get('reranked_beam_reranked', False)}")
    print(f"    Fused: {r.get('n_sentences_fused',0)} | Hedged: {r.get('n_entities_hedged',0)} | Removed: {r['n_sentences_removed']}")
    print(f"    Re-verify rounds: {r.get('n_reverify_rounds',0)}")
    print(f"    BASE: {r['baseline_report'][:100]}...")
    print(f"    CORR: {r['corrected_report'][:100]}...")


beam_rr = [r for r in results if r.get('reranked_beam_reranked', False) and r['evidence_improvement'] > 0.01]
if beam_rr:
    print(f"\n--- Beam Re-Ranked Examples ({len(beam_rr)} with Δ>0.01) ---")
    for i, r in enumerate(sorted(beam_rr, key=lambda x: x['evidence_improvement'], reverse=True)[:3]):
        print(f"  [{i+1}] {r['study_id']}: ev {r['baseline_avg_evidence']:.3f}→{r['corrected_avg_evidence']:.3f} ({r['evidence_improvement']:+.3f})")


fused_reports = [r for r in results if r.get('n_sentences_fused', 0) > 0]
if fused_reports:
    print(f"\n--- Sentence Fusion Examples ({len(fused_reports)} reports) ---")
    for i, r in enumerate(sorted(fused_reports, key=lambda x: x['evidence_improvement'], reverse=True)[:3]):
        print(f"  [{i+1}] {r['study_id']}: {r['n_sentences_fused']} fused, ev {r['baseline_avg_evidence']:.3f}→{r['corrected_avg_evidence']:.3f}")


hedged_reports = [r for r in results if r.get('n_entities_hedged', 0) > 0]
if hedged_reports:
    print(f"\n--- Hedging Examples ({len(hedged_reports)} reports) ---")
    for i, r in enumerate(sorted(hedged_reports, key=lambda x: x.get('n_entities_hedged', 0), reverse=True)[:3]):
        print(f"  [{i+1}] {r['study_id']}: {r['n_entities_hedged']} hedged, ev {r['baseline_avg_evidence']:.3f}→{r['corrected_avg_evidence']:.3f}")


blocked = [r for r in results if r.get('constrained_blocked', False)]
if blocked:
    print(f"\n--- Non-Beam Blocked ({len(blocked)} total) ---")
    for i, r in enumerate(blocked[:3]):
        print(f"  [{i+1}] {r['study_id']} — beam preserved (non-beam gain < {MIN_RERANK_GAIN})")


print("\n" + "=" * 70)
print("EVIDENCE DISTRIBUTION")
print("=" * 70)
b_scores = [r['baseline_avg_evidence'] for r in results]
c_scores = [r['corrected_avg_evidence'] for r in results]
print(f"\n{'Stat':<10} {'Baseline':>10} {'Corrected':>10}")
print("-" * 32)
for name, fn in [('Mean',np.mean),('Median',np.median),('Std',np.std),('Min',np.min),('Max',np.max)]:
    print(f"{name:<10} {fn(b_scores):>10.4f} {fn(c_scores):>10.4f}")

total_rm = int(results_df['n_sentences_removed'].sum())
total_fused = int(results_df['n_sentences_fused'].sum())
total_hedged = int(results_df['n_entities_hedged'].sum())
total_reverified = int((results_df['n_reverify_rounds'] > 0).sum())
n_modified = sum(1 for r in results if r['rerank_changed'] or r['n_sentences_removed'] > 0 or r.get('n_sentences_fused', 0) > 0)
print(f"\n📊 V3.5 Stats:")
print(f"   Beam re-ranked: {n_beam_reranked}")
print(f"   Sentences fused: {total_fused}")
print(f"   Entities hedged: {total_hedged}")
print(f"   Reports re-verified: {total_reverified}")
print(f"   Sentences removed: {total_rm}")
print(f"   Non-beam blocked: {len(blocked)}")
print(f"   Total modified: {n_modified}")

In [ ]:
corrected_df = pd.DataFrame([{
    'study_id': r['study_id'],
    'baseline_report': r['baseline_report'],
    'reranked_report': r['reranked_report'],
    'corrected_report': r['corrected_report'],
    'ground_truth': r['ground_truth'],
    'baseline_evidence': r['baseline_avg_evidence'],
    'reranked_evidence': r['reranked_avg_evidence'],
    'corrected_evidence': r['corrected_avg_evidence'],
    'rerank_changed': r['rerank_changed'],
    'reranked_strategy': r.get('reranked_strategy', 'beam'),
    'reranked_beam_reranked': r.get('reranked_beam_reranked', False),
    'constrained_blocked': r.get('constrained_blocked', False),
    'n_unique_candidates': r.get('n_unique_candidates', 1),
    'n_sentences_fused': r.get('n_sentences_fused', 0),
    'n_entities_hedged': r.get('n_entities_hedged', 0),
    'n_reverify_rounds': r.get('n_reverify_rounds', 0),
    'n_sentences_removed': r['n_sentences_removed'],
} for r in results])

corrected_path = OUTPUT_DIR / 'predictions_corrected.csv'
corrected_df.to_csv(corrected_path, index=False)
print(f"✅ Saved: {corrected_path}")

total_removed = int(results_df['n_sentences_removed'].sum())
total_fused = int(results_df['n_sentences_fused'].sum())
total_hedged = int(results_df['n_entities_hedged'].sum())
total_reverified = int((results_df['n_reverify_rounds'] > 0).sum())
n_reranked = int(results_df['rerank_changed'].sum())
n_blocked = sum(1 for r in results if r.get('constrained_blocked', False))

all_metrics = {
    'baseline': {'nlg': baseline_nlg, 'radgraph': rg_baseline_scores,
                  'safety': {'avg_evidence': float(avg_ev_b), 'support_rate': float(sr_b)}},
    'reranked': {'nlg': reranked_nlg, 'radgraph': rg_reranked_scores,
                  'safety': {'avg_evidence': float(avg_ev_r)}},
    'corrected': {'nlg': corrected_nlg, 'radgraph': rg_corrected_scores,
                   'safety': {'avg_evidence': float(avg_ev_c), 'support_rate': float(sr_c)}},
    'improvement': {
        'BLEU-4_delta': corrected_nlg['BLEU-4'] - baseline_nlg['BLEU-4'],
        'ROUGE-L_delta': corrected_nlg['ROUGE-L'] - baseline_nlg['ROUGE-L'],
        'CIDEr-D_delta': corrected_nlg['CIDEr-D'] - baseline_nlg['CIDEr-D'],
        'METEOR_delta': corrected_nlg['METEOR'] - baseline_nlg['METEOR'],
        'RG_ER_delta': rg_corrected_scores['RG_ER'] - rg_baseline_scores['RG_ER'],
        'evidence_delta': float(safety_metrics['evidence_improvement']),
        'support_rate_delta': float(safety_metrics['support_rate_improvement']),
    },
    'scas': scas, 'significance': significance,
    'config': {
        'K_candidates': K_CANDIDATES, 'remove_threshold': REMOVE_THRESHOLD,
        'min_sentences': MIN_SENTENCES, 'min_rerank_gain': MIN_RERANK_GAIN,
        'beam_alpha_evidence': BEAM_ALPHA_EVIDENCE,
        'beam_alpha_logprob': BEAM_ALPHA_LOGPROB,
        'sent_fusion_ev_threshold': SENT_FUSION_EV_THRESHOLD,
        'sent_fusion_sim_min': SENT_FUSION_SIM_MIN,
        'sent_fusion_ev_gain_min': SENT_FUSION_EV_GAIN_MIN,
        'max_fusions_per_report': MAX_FUSIONS_PER_REPORT,
        'hedging': True,
        'hedge_threshold': HEDGE_THRESHOLD,
        'max_reverification': MAX_REVERIFICATION,
        'entity_matching': 'token_overlap',
        'diversity_strategies': ['beam_search', 'mc_dropout', 'tta'],
        'strategy': 'dual_track_safe_sentence_fusion_hedging_reverify_v3.5',
    },
    'stats': {
        'n_reports': len(results), 'n_reranked': n_reranked,
        'n_beam_reranked': n_beam_reranked,
        'n_constrained_blocked': n_blocked,
        'n_fused_reports': n_fused_reports,
        'n_fused_sentences': n_fused_sentences_total,
        'n_hedged_total': total_hedged,
        'n_reverified': total_reverified,
        'n_with_removal': int((results_df['n_sentences_removed'] > 0).sum()),
        'total_sentences_removed': total_removed,
        'avg_unique_candidates': float(results_df['n_unique_candidates'].mean()),
    }
}

metrics_path = OUTPUT_DIR / 'module4_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(all_metrics, f, indent=2)
print(f"✅ Saved: {metrics_path}")

detailed_path = OUTPUT_DIR / 'module4_detailed_results.json'
with open(detailed_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f"✅ Saved: {detailed_path}")

zip_path = '/kaggle/working/module4_outputs'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f"✅ Saved: {zip_path}.zip")


def pct(d, b):
    return d/b*100 if b else 0

b4b, b4r, b4c = baseline_nlg['BLEU-4'], reranked_nlg['BLEU-4'], corrected_nlg['BLEU-4']
rlb, rlr, rlc = baseline_nlg['ROUGE-L'], reranked_nlg['ROUGE-L'], corrected_nlg['ROUGE-L']
cdb, cdr, cdc = baseline_nlg['CIDEr-D'], reranked_nlg['CIDEr-D'], corrected_nlg['CIDEr-D']
mtb, mtr, mtc = baseline_nlg['METEOR'], reranked_nlg['METEOR'], corrected_nlg['METEOR']
rgb, rgr, rgc = rg_baseline_scores['RG_ER'], rg_reranked_scores['RG_ER'], rg_corrected_scores['RG_ER']

print(f"\n{'='*85}")
print(f"MODULE 4 COMPLETE — V3.5 Dual-Track + SAFE Fusion + Hedging + Re-verify")
print(f"{'='*85}")
print(f)

print(f"\nFull Pipeline:")
print(f"  ✅ Module 1: VG-SCRRG V4.1 (AUROC=0.8619, MIMIC-CXR)")
print(f"  ✅ Module 2: R2Gen baseline (BLEU-4 = {b4b:.4f})")
print(f"  ✅ Module 3: RadGraph entity extraction (RG_ER = {rgb:.4f})")
print(f"  ✅ Module 4: Dual-Track + SAFE Fusion + Hedging + Re-verify V3.5")

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
try:
    import seaborn as sns
    _HAS_SNS = True
except ImportError:
    _HAS_SNS = False



IEEE_SINGLE_COL = 3.5    
IEEE_DOUBLE_COL = 7.16   

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'serif'],
    'font.size': 8,
    'axes.titlesize': 9,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.02,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linewidth': 0.5,
    'grid.linestyle': '--',
    'axes.linewidth': 0.6,
    'lines.linewidth': 1.2,
    'patch.linewidth': 0.5,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

C_BLUE = '
C_RED = '
C_GREEN = '
C_PURPLE = '
C_ORANGE = '
C_GRAY = '
C_TEAL = '

FIG_DIR = str(OUTPUT_DIR / 'figures')
os.makedirs(FIG_DIR, exist_ok=True)



fig, ax = plt.subplots(figsize=(IEEE_DOUBLE_COL, 3.5))

nlg_keys = ['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4', 'ROUGE-L', 'CIDEr-D', 'METEOR']
b_vals = [baseline_nlg[k] for k in nlg_keys]
r_vals = [reranked_nlg[k] for k in nlg_keys]
c_vals = [corrected_nlg[k] for k in nlg_keys]

x = np.arange(len(nlg_keys))
w = 0.24

bars_b = ax.bar(x - w, b_vals, w, color=C_BLUE, label='Baseline (R2Gen)', edgecolor='white', linewidth=0.5)
bars_r = ax.bar(x, r_vals, w, color=C_ORANGE, label='Re-ranked', edgecolor='white', linewidth=0.5)
bars_c = ax.bar(x + w, c_vals, w, color=C_GREEN, label='VG-SCRRG Corrected', edgecolor='white', linewidth=0.5)


for bars in [bars_b, bars_r, bars_c]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.003, f'{h:.3f}',
                ha='center', va='bottom', fontsize=6.5, rotation=0)

ax.set_xticks(x)
ax.set_xticklabels(nlg_keys)
ax.set_ylabel('Score')
ax.set_title('NLG Metrics: Baseline → Re-ranked → VG-SCRRG Corrected (MIMIC-CXR, MIMIC-CXR)')
ax.legend(fontsize=8, framealpha=0.9, loc='upper right')
ax.set_ylim([0, max(max(b_vals), max(r_vals), max(c_vals)) + 0.05])

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_m4_nlg_comparison.png'))
fig.savefig(os.path.join(FIG_DIR, 'fig_m4_nlg_comparison.pdf'))
plt.show()
print("✅ Saved: fig_m4_nlg_comparison.{png,pdf}")



fig, axes = plt.subplots(1, 3, figsize=(IEEE_DOUBLE_COL, 3.0))


rg_labels = ['RG_E', 'RG_ER', 'RG_bar_ER']
rg_b = [rg_baseline_scores[k] for k in rg_labels]
rg_r = [rg_reranked_scores[k] for k in rg_labels]
rg_c = [rg_corrected_scores[k] for k in rg_labels]
x_rg = np.arange(len(rg_labels))

axes[0].bar(x_rg - w, rg_b, w, color=C_BLUE, label='Baseline', edgecolor='white')
axes[0].bar(x_rg, rg_r, w, color=C_ORANGE, label='Re-ranked', edgecolor='white')
axes[0].bar(x_rg + w, rg_c, w, color=C_GREEN, label='Corrected', edgecolor='white')

for i, (b, r, c) in enumerate(zip(rg_b, rg_r, rg_c)):
    axes[0].text(i - w, b + 0.003, f'{b:.3f}', ha='center', fontsize=7, color=C_BLUE)
    axes[0].text(i, r + 0.003, f'{r:.3f}', ha='center', fontsize=7, color=C_ORANGE)
    axes[0].text(i + w, c + 0.003, f'{c:.3f}', ha='center', fontsize=7, color=C_GREEN)

axes[0].set_xticks(x_rg)
axes[0].set_xticklabels(['Entity F1', 'E+R F1', 'Macro E+R'])
axes[0].set_ylabel('F1-RadGraph')
axes[0].set_title('(a) F1-RadGraph')
axes[0].legend(fontsize=7, framealpha=0.9)


ev_names = ['Baseline', 'Re-ranked', 'Corrected']
ev_vals = [avg_ev_b, avg_ev_r, avg_ev_c]
ev_colors = [C_BLUE, C_ORANGE, C_GREEN]

bars_ev = axes[1].bar(ev_names, ev_vals, color=ev_colors, edgecolor='white', linewidth=0.5, width=0.55)
for bar, val in zip(bars_ev, ev_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{val:.4f}', ha='center', fontsize=8)


delta_ev = avg_ev_c - avg_ev_b
axes[1].annotate(f'Δ = {delta_ev:+.4f}\n(p = {significance["ttest_p"]:.1e})',
                 xy=(2, avg_ev_c), xytext=(1.5, avg_ev_c + 0.02),
                 fontsize=8, ha='center',
                 arrowprops=dict(arrowstyle='->', color=C_GRAY, lw=0.8),
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor=C_GRAY))
axes[1].set_ylabel('Avg Evidence Score')
axes[1].set_title('(b) Visual Evidence')


scas_labels = ['Overall', f"Normal\n(n={scas['n_normal']})", f"Abnormal\n(n={scas['n_abnormal']})"]
scas_b = [scas['scas_baseline'], scas['scas_normal_baseline'], scas['scas_abnormal_baseline']]
scas_c = [scas['scas_corrected'], scas['scas_normal_corrected'], scas['scas_abnormal_corrected']]
x_sc = np.arange(3)

axes[2].bar(x_sc - 0.17, scas_b, 0.3, color=C_BLUE, label='Baseline', edgecolor='white')
axes[2].bar(x_sc + 0.17, scas_c, 0.3, color=C_GREEN, label='Corrected', edgecolor='white')

for i, (b, c) in enumerate(zip(scas_b, scas_c)):
    axes[2].text(i - 0.17, b + 0.003, f'{b:.3f}', ha='center', fontsize=7, color=C_BLUE)
    axes[2].text(i + 0.17, c + 0.003, f'{c:.3f}', ha='center', fontsize=7, color=C_GREEN)

axes[2].set_xticks(x_sc)
axes[2].set_xticklabels(scas_labels, fontsize=8)
axes[2].set_ylabel('SCAS')
axes[2].set_title('(c) Supported Content (SCAS)')
axes[2].legend(fontsize=7, framealpha=0.9)

plt.tight_layout(w_pad=2)
fig.savefig(os.path.join(FIG_DIR, 'fig_m4_clinical_accuracy.png'))
fig.savefig(os.path.join(FIG_DIR, 'fig_m4_clinical_accuracy.pdf'))
plt.show()
print("✅ Saved: fig_m4_clinical_accuracy.{png,pdf}")



fig, ax = plt.subplots(figsize=(IEEE_DOUBLE_COL, 3.0))

b_ev = [r['baseline_avg_evidence'] for r in results]
c_ev = [r['corrected_avg_evidence'] for r in results]

bins_ev = np.linspace(min(min(b_ev), min(c_ev)) - 0.02, max(max(b_ev), max(c_ev)) + 0.02, 40)
ax.hist(b_ev, bins=bins_ev, alpha=0.6, color=C_BLUE, density=True, edgecolor='white',
        linewidth=0.3, label=f'Baseline (μ={np.mean(b_ev):.4f})')
ax.hist(c_ev, bins=bins_ev, alpha=0.6, color=C_GREEN, density=True, edgecolor='white',
        linewidth=0.3, label=f'Corrected (μ={np.mean(c_ev):.4f})')

ax.axvline(np.mean(b_ev), color=C_BLUE, linestyle='--', linewidth=1.5, alpha=0.8)
ax.axvline(np.mean(c_ev), color=C_GREEN, linestyle='--', linewidth=1.5, alpha=0.8)


ax.annotate(f'Δμ = {np.mean(c_ev) - np.mean(b_ev):+.4f}',
            xy=((np.mean(b_ev)+np.mean(c_ev))/2, ax.get_ylim()[1]*0.85),
            fontsize=9, ha='center', weight='bold',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', edgecolor=C_GRAY))

ax.set_xlabel('Average Evidence Score')
ax.set_ylabel('Density')
ax.set_title('Evidence Score Distribution: Baseline vs VG-SCRRG Corrected')
ax.legend(fontsize=8, framealpha=0.9)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_m4_evidence_distribution.png'))
fig.savefig(os.path.join(FIG_DIR, 'fig_m4_evidence_distribution.pdf'))
plt.show()
print("✅ Saved: fig_m4_evidence_distribution.{png,pdf}")



fig, ax = plt.subplots(figsize=(IEEE_DOUBLE_COL, 3.5))

improvements = [r['corrected_avg_evidence'] - r['baseline_avg_evidence'] for r in results]


colors_scatter = [C_GREEN if d > 0 else C_RED if d < 0 else C_GRAY for d in improvements]
ax.scatter(b_ev, improvements, c=colors_scatter, s=12, alpha=0.5, edgecolors='none')
ax.axhline(0, color=C_GRAY, linestyle='-', linewidth=1.0)


z = np.polyfit(b_ev, improvements, 1)
p_fit = np.poly1d(z)
x_fit = np.linspace(min(b_ev), max(b_ev), 100)
ax.plot(x_fit, p_fit(x_fit), color=C_PURPLE, linestyle='--', linewidth=1.5,
        label=f'Trend (slope = {z[0]:.3f})')

n_improved = sum(1 for d in improvements if d > 0)
n_degraded = sum(1 for d in improvements if d < 0)
n_same = len(improvements) - n_improved - n_degraded

ax.text(0.02, 0.98, f'Improved: {n_improved}\nDegraded: {n_degraded}\nUnchanged: {n_same}',
        transform=ax.transAxes, fontsize=8, va='top',
        bbox=dict(boxstyle='round', facecolor='white', edgecolor=C_GRAY, alpha=0.9))

ax.set_xlabel('Baseline Average Evidence')
ax.set_ylabel('Evidence Improvement (Corrected − Baseline)')
ax.set_title('Per-Sample Evidence Change')
ax.legend(fontsize=8, loc='lower right')

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_m4_evidence_scatter.png'))
fig.savefig(os.path.join(FIG_DIR, 'fig_m4_evidence_scatter.pdf'))
plt.show()
print("✅ Saved: fig_m4_evidence_scatter.{png,pdf}")

print("\n✅ Cell 11a complete — 4 main figures saved")

In [ ]:
fig, ax = plt.subplots(figsize=(IEEE_DOUBLE_COL, 3.5))


n_total = len(results)


ev_baseline = np.mean([r['baseline_avg_evidence'] for r in results])

ev_reranked = np.mean([r['reranked_avg_evidence'] for r in results])

ev_corrected = np.mean([r['corrected_avg_evidence'] for r in results])


reranked_mask = np.array([r['rerank_changed'] for r in results])
fused_mask = np.array([r.get('n_sentences_fused', 0) > 0 for r in results])
hedged_mask = np.array([r.get('n_entities_hedged', 0) > 0 for r in results])
removed_mask = np.array([r['n_sentences_removed'] > 0 for r in results])
reverify_mask = np.array([r.get('n_reverify_rounds', 0) > 0 for r in results])

ablation_labels = [
    f'Baseline\n(R2Gen)',
    f'+ Re-ranking\n(n={reranked_mask.sum()})',
    f'+ Sentence\nFusion (n={fused_mask.sum()})',
    f'+ Hedging\n(n={hedged_mask.sum()})',
    f'+ Conservative\nRemoval (n={removed_mask.sum()})',
    f'+ Re-verification\n(n={reverify_mask.sum()})',
    f'Full Pipeline\n(V3.5)',
]






ev_after_rerank = float(np.mean([r['reranked_avg_evidence'] for r in results]))








unmodified = [r for r in results if not r['rerank_changed'] and r['n_sentences_removed'] == 0
              and r.get('n_sentences_fused', 0) == 0 and r.get('n_entities_hedged', 0) == 0]
ev_unmodified = float(np.mean([r['baseline_avg_evidence'] for r in unmodified])) if unmodified else ev_baseline

ablation_vals = [
    ev_baseline,          
    ev_after_rerank,      
    ev_corrected,         
]
ablation_labels = [
    f'Baseline\n(R2Gen)',
    f'+ Re-ranking\n(n={reranked_mask.sum()})',
    f'Full Pipeline\n(V3.5)',
]

colors_ab = [C_BLUE, C_ORANGE, C_GREEN]
bars = ax.bar(range(len(ablation_labels)), ablation_vals, color=colors_ab,
              edgecolor='white', linewidth=0.5, width=0.65)

for bar, val in zip(bars, ablation_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=7.5)


for i in range(1, len(ablation_vals)):
    delta = ablation_vals[i] - ablation_vals[i-1]
    if delta > 0.0005:
        mid_y = (ablation_vals[i] + ablation_vals[i-1]) / 2
        ax.annotate('', xy=(i, ablation_vals[i]*0.998), xytext=(i-1, ablation_vals[i-1]*0.998),
                    arrowprops=dict(arrowstyle='->', color=C_GRAY, lw=0.6, alpha=0.5))

ax.set_xticks(range(len(ablation_labels)))
ax.set_xticklabels(ablation_labels, fontsize=7, rotation=0, ha='center')
ax.set_ylabel('Average Evidence Score')
ax.set_title('Ablation: VG-SCRRG V3.5 Pipeline Component Contributions')
ax.set_ylim([ev_baseline - 0.01, ev_corrected + 0.02])

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_m4_ablation.png'))
fig.savefig(os.path.join(FIG_DIR, 'fig_m4_ablation.pdf'))
plt.show()
print("✅ Saved: fig_m4_ablation.{png,pdf}")



if chexbert_baseline is not None and chexbert_corrected is not None:
    fig, ax = plt.subplots(figsize=(IEEE_DOUBLE_COL, 4.5))

    pathology_names = list(chexbert_baseline['per_class'].keys())
    b_f1 = [chexbert_baseline['per_class'][p] for p in pathology_names]
    c_f1 = [chexbert_corrected['per_class'][p] for p in pathology_names]

    y = np.arange(len(pathology_names))
    h = 0.35

    ax.barh(y + h/2, b_f1, h, color=C_BLUE, label='Baseline', edgecolor='white', linewidth=0.3)
    ax.barh(y - h/2, c_f1, h, color=C_GREEN, label='VG-SCRRG Corrected', edgecolor='white', linewidth=0.3)

    
    for i, (b, c) in enumerate(zip(b_f1, c_f1)):
        delta = c - b
        marker = '↑' if delta > 0.005 else '↓' if delta < -0.005 else '='
        color = C_GREEN if delta > 0.005 else C_RED if delta < -0.005 else C_GRAY
        ax.text(max(b, c) + 0.015, i, f'Δ={delta:+.3f} {marker}',
                va='center', fontsize=7, color=color)

    ax.set_yticks(y)
    ax.set_yticklabels(pathology_names, fontsize=8)
    ax.set_xlabel('F1 Score')
    ax.set_title('CheXbert Per-Class F1: Baseline vs VG-SCRRG Corrected')
    ax.legend(fontsize=8, loc='lower right', framealpha=0.9)
    ax.invert_yaxis()
    ax.set_xlim([0, max(max(b_f1), max(c_f1)) + 0.10])

    
    ax.text(0.98, 0.02,
            f"Micro-F1: {chexbert_baseline['micro_f1']:.3f}→{chexbert_corrected['micro_f1']:.3f}\n"
            f"Macro-F1: {chexbert_baseline['macro_f1']:.3f}→{chexbert_corrected['macro_f1']:.3f}\n"
            f"CE-5 F1:  {chexbert_baseline['ce5_f1']:.3f}→{chexbert_corrected['ce5_f1']:.3f}",
            transform=ax.transAxes, fontsize=8, va='bottom', ha='right',
            bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor=C_GRAY))

    plt.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, 'fig_m4_chexbert_per_class.png'))
    fig.savefig(os.path.join(FIG_DIR, 'fig_m4_chexbert_per_class.pdf'))
    plt.show()
    print("✅ Saved: fig_m4_chexbert_per_class.{png,pdf}")
else:
    print("⚠️ CheXbert not available — skipping per-class F1 plot")



fig, (ax_hist, ax_forest) = plt.subplots(1, 2, figsize=(IEEE_DOUBLE_COL, 3.0), gridspec_kw={'width_ratios': [2, 1]})


ax_hist.hist(boot_means, bins=50, color=C_BLUE, alpha=0.7, edgecolor='white', linewidth=0.3)
ax_hist.axvline(mean_diff, color=C_RED, linestyle='-', linewidth=2, label=f'Mean Δ = {mean_diff:.4f}')
ax_hist.axvline(ci_low, color=C_PURPLE, linestyle='--', linewidth=1.2, label=f'95% CI low = {ci_low:.4f}')
ax_hist.axvline(ci_high, color=C_PURPLE, linestyle='--', linewidth=1.2, label=f'95% CI high = {ci_high:.4f}')
ax_hist.axvline(0, color=C_GRAY, linestyle=':', linewidth=1.0, alpha=0.8)

ax_hist.set_xlabel('Mean Evidence Improvement (Corrected − Baseline)')
ax_hist.set_ylabel('Count')
ax_hist.set_title('(a) Bootstrap Distribution (n=10,000)')
ax_hist.legend(fontsize=7, framealpha=0.9)


tests = {
    'Evidence\nImprovement': (mean_diff, ci_low, ci_high),
}
y_pos = [0]
ax_forest.errorbar([mean_diff], y_pos, xerr=[[mean_diff - ci_low], [ci_high - mean_diff]],
                    fmt='D', color=C_BLUE, markersize=8, capsize=6, linewidth=2, capthick=2)
ax_forest.axvline(0, color=C_GRAY, linestyle=':', linewidth=1.0)


test_text = (
    f"Paired t: p={significance['ttest_p']:.2e}\n"
    f"Wilcoxon: p={significance['wilcoxon_p']:.2e}\n"
    f"Cohen's d: {significance['cohens_d']:.3f} ({significance['effect_size']})\n"
    f"Bootstrap CI: [{ci_low:.4f}, {ci_high:.4f}]"
)
ax_forest.text(0.95, 0.95, test_text, transform=ax_forest.transAxes,
               fontsize=7.5, va='top', ha='right',
               bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor=C_GRAY))

ax_forest.set_yticks(y_pos)
ax_forest.set_yticklabels(['Δ Evidence'])
ax_forest.set_xlabel('Effect Size')
ax_forest.set_title('(b) Confidence Interval')

plt.tight_layout(w_pad=2)
fig.savefig(os.path.join(FIG_DIR, 'fig_m4_bootstrap_ci.png'))
fig.savefig(os.path.join(FIG_DIR, 'fig_m4_bootstrap_ci.pdf'))
plt.show()
print("✅ Saved: fig_m4_bootstrap_ci.{png,pdf}")



fig, ax = plt.subplots(figsize=(IEEE_SINGLE_COL, 3.5))

component_counts = {
    f'Re-ranked\n({reranked_mask.sum()})': reranked_mask.sum(),
    f'Sentence Fusion\n({fused_mask.sum()})': fused_mask.sum(),
    f'Hedging\n({hedged_mask.sum()})': hedged_mask.sum(),
    f'Removal\n({removed_mask.sum()})': removed_mask.sum(),
    f'Re-verification\n({reverify_mask.sum()})': reverify_mask.sum(),
}

active = {k: v for k, v in component_counts.items() if v > 0}

if active:
    pie_colors = [C_ORANGE, '
    wedges, texts, autotexts = ax.pie(
        active.values(), labels=active.keys(), colors=pie_colors,
        autopct='%1.1f%%', startangle=90, pctdistance=0.82,
        textprops={'fontsize': 8}
    )
    for t in autotexts:
        t.set_fontsize(7)
    ax.set_title(f'Pipeline Component Activity\n(n={n_total} reports)')
else:
    ax.text(0.5, 0.5, 'No components active', transform=ax.transAxes,
            ha='center', fontsize=9)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_m4_component_activity.png'))
fig.savefig(os.path.join(FIG_DIR, 'fig_m4_component_activity.pdf'))
plt.show()
print("✅ Saved: fig_m4_component_activity.{png,pdf}")



fig, axes = plt.subplots(3, 1, figsize=(IEEE_DOUBLE_COL, 8.0))

by_improvement = sorted(results, key=lambda x: x['evidence_improvement'], reverse=True)
for panel_idx, r in enumerate(by_improvement[:3]):
    ax = axes[panel_idx]
    ax.axis('off')

    study = r['study_id']
    baseline = r['baseline_report'][:200]
    corrected = r['corrected_report'][:200]
    gt_text = r['ground_truth'][:200]
    ev_b = r['baseline_avg_evidence']
    ev_c = r['corrected_avg_evidence']
    delta = r['evidence_improvement']
    strat = r.get('reranked_strategy', 'beam')
    n_fused = r.get('n_sentences_fused', 0)
    n_hedged = r.get('n_entities_hedged', 0)
    n_removed = r['n_sentences_removed']
    n_reverify = r.get('n_reverify_rounds', 0)

    text = (
        f"Case {panel_idx+1}: {study}\n"
        f"Evidence: {ev_b:.4f} → {ev_c:.4f} (Δ = {delta:+.4f})  |  "
        f"Strategy: {strat}  |  Fused: {n_fused}  |  Hedged: {n_hedged}  |  "
        f"Removed: {n_removed}  |  Re-verify: {n_reverify}\n\n"
        f"Ground Truth:\n{gt_text}...\n\n"
        f"Baseline (R2Gen):\n{baseline}...\n\n"
        f"VG-SCRRG Corrected:\n{corrected}..."
    )

    ax.text(0.02, 0.98, text, transform=ax.transAxes, fontsize=8,
            va='top', ha='left', family='monospace',
            bbox=dict(boxstyle='round', facecolor='

    
    color = C_GREEN if delta > 0.01 else C_RED if delta < -0.01 else C_GRAY
    ax.add_patch(mpatches.FancyBboxPatch(
        (0.0, 0.0), 0.005, 1.0, transform=ax.transAxes,
        boxstyle="round,pad=0", facecolor=color, alpha=0.7
    ))

plt.suptitle('Qualitative Case Study: Top-3 Evidence Improvements', fontsize=9, y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_m4_case_study.png'))
fig.savefig(os.path.join(FIG_DIR, 'fig_m4_case_study.pdf'))
plt.show()
print("✅ Saved: fig_m4_case_study.{png,pdf}")




fig, ax = plt.subplots(figsize=(IEEE_SINGLE_COL, 2.8))

scas_f1_labels = ['SCAS', 'SCAS-F1']
scas_b_vals = [scas['scas_baseline'], scas_f1_baseline]
scas_c_vals = [scas['scas_corrected'], scas_f1_corrected]

x_sf = np.arange(len(scas_f1_labels))
ax.bar(x_sf - 0.17, scas_b_vals, 0.3, color=C_BLUE, label='Baseline', edgecolor='white')
ax.bar(x_sf + 0.17, scas_c_vals, 0.3, color=C_GREEN, label='Corrected', edgecolor='white')

for i, (b, c) in enumerate(zip(scas_b_vals, scas_c_vals)):
    ax.text(i - 0.17, b + 0.005, f'{b:.4f}', ha='center', fontsize=8, color=C_BLUE)
    ax.text(i + 0.17, c + 0.005, f'{c:.4f}', ha='center', fontsize=8, color=C_GREEN)
    delta = c - b
    ax.annotate(f'Δ={delta:+.4f}', xy=(i, max(b, c) + 0.015), fontsize=8, ha='center',
                color=C_GREEN if delta > 0 else C_RED)

ax.set_xticks(x_sf)
ax.set_xticklabels(scas_f1_labels)
ax.set_ylabel('Score')
ax.set_title('Supported Content: SCAS vs SCAS-F1 (Deletion-Resistant)')
ax.legend(fontsize=8, framealpha=0.9)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_m4_scas_f1.png'))
fig.savefig(os.path.join(FIG_DIR, 'fig_m4_scas_f1.pdf'))
plt.show()
print("✅ Saved: fig_m4_scas_f1.{png,pdf}")




print("\n" + "=" * 70)
print("MODULE 4 — PUBLICATION FIGURES SUMMARY")
print("=" * 70)
print(f)